# Chapter 20: Check

> *"A type system is a tractable syntactic method for proving the
> absence of certain program behaviours by classifying phrases according
> to the kinds of values they compute."* — Benjamin C. Pierce, Types and
> Programming Languages (2002)

------------------------------------------------------------------------

### 20.1 Semantic Analysis: Beyond Syntactic Structure

In Chapters 18 and 19, we built the compiler's syntax engine. The
lexical scanner converted source characters into tokens, and the
recursive descent parser organised those tokens into an Abstract Syntax
Tree (AST).

However, structural validity does not imply semantic correctness.
Consider the following syntactically flawless **Faberge** snippet:

```
let compute = fn (x: Int): Int => {
    let message = "Total score: ";
    let flag = true;
    (message * flag) + undefined_variable(x)
};
```

If you feed this text into our Chapter 19 parser, it constructs an AST
without raising a single error. The grammar productions are satisfied:
`message * flag` is a syntactically valid `BinaryExprAST`, and
`undefined_variable(x)` is a valid `CallExprAST`.

Yet the program is nonsensical:

1.  **Type Incompatibility:** What does it mean to multiply a `String`
    by a `Bool`?
2.  **Undeclared Symbols:** The identifier `undefined_variable` was
    never declared in this scope or any enclosing scope.
3.  **Return Type Violation:** The function signature promises to return
    an `Int`, but its body evaluates to an illegal compound expression.

<figure id="fig:semantic-analysis">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI0NDBwdCIgaGVpZ2h0PSIxOTZwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA0NDAuMDAgMTk2LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE5
MikiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXN0IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPmFzdDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iNDMyLC0xODggMCwtMTg4IDAsLTE0NSA0MzIsLTE0NSA0MzIsLTE4OCIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0xNzUuNSIgZm9udC1mYW1pbHk9IkNv
dXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAi
PkFTVCAoQWJzdHJhY3QgU3ludGF4IFRyZWUgZnJvbSBDaGFwdGVyIDE5KTwvdGV4dD4KPHRleHQg
dGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMTUyLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5Qcm9n
cmFtQVNUICYjNDU7Jmd0OyBMZXRTdG10QVNUICYjNDU7Jmd0OyBMYW1iZGFFeHByQVNUICYjNDU7
Jmd0OyBCbG9ja0V4cHJBU1QgLi4uPC90ZXh0Pgo8L2c+CjwhLS0gc2VtYW50aWMgLS0+CjxnIGlk
PSJub2RlMiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+c2VtYW50aWM8L3RpdGxlPgo8cG9seWdvbiBm
aWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjQzMiwtMTIzIDAsLTEyMyAwLC01OCA0
MzIsLTU4IDQzMiwtMTIzIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTEx
MCIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2Ui
IGZvbnQtc2l6ZT0iMTAuMDAiPlNUQUdFIDM6IFNlbWFudGljIEFuYWx5c2lzIGFuZCBTdGF0aWMg
VHlwZSBDaGVja2luZyAoVGhpcyBDaHBhdGVyKTwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0
YXJ0IiB4PSI4IiB5PSItOTkiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMg
TW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4xLiBTY29wZSBSZXNvbHV0aW9uIChC
aW5kIGlkZW50aWZpZXJzIHRvIFN5bWJvbCBUYWJsZSBlbnRyaWVzKTwvdGV4dD4KPHRleHQgdGV4
dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItODgiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywg
RGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4yLiBTdGF0aWMg
VHlwZSBDaGVja2luZyAoU3ludGhlc2lzZSBhbmQgdmVyaWZ5IG9wZXJhbmQgdHlwZXMpPC90ZXh0
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii03NyIgZm9udC1mYW1pbHk9IkNv
dXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAi
PjMuIENvbnRyb2wmIzQ1O0Zsb3cgYW5kIFJldHVybiBWZXJpZmljYXRpb248L3RleHQ+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTY2IiBmb250LWZhbWlseT0iQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+NC4gRGVj
b3JhdGUgQVNUIHdpdGggVmVyaWZpZWQgVHlwZXM8L3RleHQ+CjwvZz4KPCEtLSBhc3QmIzQ1OyZn
dDtzZW1hbnRpYyAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5hc3QmIzQ1
OyZndDtzZW1hbnRpYzwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9
Ik0yMTYsLTE0NC44OUMyMTYsLTE0MS4xOSAyMTYsLTEzNy4yMiAyMTYsLTEzMy4xOCIvPgo8cG9s
eWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyMTkuNSwtMTMzLjA4IDIx
NiwtMTIzLjA4IDIxMi41LC0xMzMuMDggMjE5LjUsLTEzMy4wOCIvPgo8L2c+CjwhLS0gY29kZSAt
LT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5jb2RlPC90aXRsZT4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjE2IiB5PSItMTUuNSIgZm9udC1mYW1pbHk9IkNvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkRl
Y29yYXRlZCAvIFZlcmlmaWVkIFR5cGVkIEFTVCAoUmVhZHkgZm9yIENvZGUgR2VuZXJhdGlvbik8
L3RleHQ+CjwvZz4KPCEtLSBzZW1hbnRpYyYjNDU7Jmd0O2NvZGUgLS0+CjxnIGlkPSJlZGdlMiIg
Y2xhc3M9ImVkZ2UiPgo8dGl0bGU+c2VtYW50aWMmIzQ1OyZndDtjb2RlPC90aXRsZT4KPHBhdGgg
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIxNiwtNTcuOTZDMjE2LC01NC4xMSAyMTYs
LTUwLjE5IDIxNiwtNDYuNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSIyMTkuNSwtNDYuMjIgMjE2LC0zNi4yMiAyMTIuNSwtNDYuMjIgMjE5LjUsLTQ2LjIy
Ii8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 20–1: Detailed Overview of Semantic
Analysis</figcaption>
</figure>

**Semantic Analysis** is the compiler pass that enforces the contextual
and mathematical rules of the language (see Figure 20–1). In this
chapter, we build **Stage 3** of our compiler project: a **Static Type
Checker and Semantic Analyser** in Modern C++20.

### 20.2 Lexical Scoping and the Scoped Symbol Table

To check whether an identifier is legal, the compiler must track every
variable, parameter, and function declaration in existence at any point
during program execution. This tracking structure is the **Symbol
Table** (or **Environment**).

#### 20.2.1 Lexical Scoping Rules

Faberge—like C++, TypeScript, Rust, and Scheme—uses **Lexical (Static)
Scoping**. In a lexically scoped language:

1.  The scope of an identifier is determined entirely at compile time by
    its textual position in the source code.
2.  Inner blocks inherit all bindings declared in enclosing outer
    blocks.
3.  **Variable Shadowing:** An inner scope may declare a variable with
    the exact same name as an outer variable. The inner declaration
    "shadows" (hides) the outer declaration for the duration of the
    inner block, without destroying the outer variable.
4.  Outer scopes cannot see variables declared inside nested inner
    scopes.

See Figure 20–2 for an illustration of lexical scoping lookup.

<figure id="fig:scopes">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCAyMCA4NjAgMjMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMzAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0ic2NvcGVTaGFkb3ciIHg9Ii01JSIgeT0i
LTUlIiB3aWR0aD0iMTEwJSIgaGVpZ2h0PSIxMjAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBkeD0i
MCIgZHk9IjMiIHN0ZERldmlhdGlvbj0iMyIgZmxvb2QtY29sb3I9InZhcigtLXN2Zy1mZykiIGZs
b29kLW9wYWNpdHk9IjAuNSIvPgogICAgPC9maWx0ZXI+CiAgPC9kZWZzPgogIDwhLS0gR2xvYmFs
IFNjb3BlIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg0MCwgMzApIj4KICAgIDxy
ZWN0IHg9IjAiIHk9IjAiIHdpZHRoPSIyNDAiIGhlaWdodD0iMjAwIiByeD0iOCIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigt
LXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI3Njb3BlU2hhZG93KSIvPgog
ICAgPHRleHQgeD0iMTIwIiB5PSIyMyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIx
NCIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlNjb3BlIDA6IEdsb2Jh
bCBFbnZpcm9ubWVudDwvdGV4dD4KICAgIDwhLS0gR2xvYmFsIFN5bWJvbHMgLS0+CiAgICA8dGV4
dCB4PSIyMCIgeT0iNjUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPnBpIDog
RmxvYXQ8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iOTAiIGZpbGw9InZhcigtLXN2Zy1mZyki
IGZvbnQtc2l6ZT0iMTQiPmdsb2JhbF9saW1pdCA6IEludDwvdGV4dD4KICAgIDx0ZXh0IHg9IjIw
IiB5PSIxMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPmZhY3RvcmlhbCA6
IChJbnQgLSZndDsgSW50KTwvdGV4dD4KICAgIDxsaW5lIHgxPSIyMCIgeTE9IjEzNSIgeDI9IjIy
MCIgeTI9IjEzNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgog
ICAgPHRleHQgeD0iMjAiIHk9IjE2MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIx
NCI+UGFyZW50OiBudWxscHRyIChSb290KTwvdGV4dD4KICA8L2c+CiAgPCEtLSBBcnJvdyBmcm9t
IEZ1bmMgU2NvcGUgdG8gR2xvYmFsIFNjb3BlIC0tPgogIDxwYXRoIGQ9Ik0gMzMwIDE0MCBMIDI5
NSAxNDAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyLjUiIGZpbGw9Im5v
bmUiLz4KICA8cG9seWdvbiBwb2ludHM9IjI5MCwxNDAgMjk4LDEzNSAyOTgsMTQ1IiBmaWxsPSJ2
YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iMzEwIiB5PSIxMzAiIGZpbGw9InZhcigtLXN2Zy1m
ZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPnBhcmVudDwvdGV4dD4KICA8
IS0tIEZ1bmN0aW9uIFNjb3BlIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSgzNDAs
IDMwKSI+CiAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMjQwIiBoZWlnaHQ9IjIwMCIgcng9
IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0idXJsKCNzY29w
ZVNoYWRvdykiLz4KICAgIDx0ZXh0IHg9IjEyMCIgeT0iMjMiIGZpbGw9InZhcigtLXN2Zy1mZyki
IGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5T
Y29wZSAxOiBGdW5jdGlvbiAnY2FsY3VsYXRlJzwvdGV4dD4KICAgIDwhLS0gRnVuY3Rpb24gU3lt
Ym9scyAtLT4KICAgIDx0ZXh0IHg9IjIwIiB5PSI2NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9u
dC1zaXplPSIxNCI+eCA6IEludCAoUGFyYW0pPC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9Ijkw
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij55IDogRmxvYXQgKFBhcmFtKTwv
dGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSIxMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQt
c2l6ZT0iMTQiPm11bHRpcGxpZXIgOiBJbnQgPSAxMDwvdGV4dD4KICAgIDxsaW5lIHgxPSIyMCIg
eTE9IjEzNSIgeDI9IjIyMCIgeTI9IjEzNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Ut
d2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0iMjAiIHk9IjE2MCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSIgZm9udC1zaXplPSIxNCI+SW5oZXJpdHM6IHBpLCBmYWN0b3JpYWw8L3RleHQ+CiAgPC9nPgog
IDwhLS0gQXJyb3cgZnJvbSBCbG9jayBTY29wZSB0byBGdW5jIFNjb3BlIC0tPgogIDxwYXRoIGQ9
Ik0gNjMwIDE0MCBMIDU5NSAxNDAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRo
PSIyLjUiIGZpbGw9Im5vbmUiLz4KICA8cG9seWdvbiBwb2ludHM9IjU5MCwxNDAgNTk4LDEzNSA1
OTgsMTQ1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iNjEwIiB5PSIxMzAiIGZp
bGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPnBh
cmVudDwvdGV4dD4KICA8IS0tIEJsb2NrIFNjb3BlIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09InRy
YW5zbGF0ZSg2NDAsIDMwKSI+CiAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMjMwIiBoZWln
aHQ9IjIwMCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEt
ZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRl
cj0idXJsKCNzY29wZVNoYWRvdykiLz4KICAgIDx0ZXh0IHg9IjExNSIgeT0iMjMiIGZpbGw9InZh
cigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hv
cj0ibWlkZGxlIj5TY29wZSAyOiBOZXN0ZWQgQmxvY2sgeyAuLi4gfTwvdGV4dD4KICAgIDwhLS0g
QmxvY2sgU3ltYm9scyAtLT4KICAgIDx0ZXh0IHg9IjIwIiB5PSI2NSIgZmlsbD0idmFyKC0tc3Zn
LWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiPnggOiBTdHJpbmcgPSAic2hh
ZG93IjwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI5MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1zaXplPSIxNCI+dGVtcF9vZmZzZXQgOiBJbnQgPSA0MjwvdGV4dD4KICAgIDxsaW5lIHgx
PSIyMCIgeTE9IjEzNSIgeDI9IjIxMCIgeTI9IjEzNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBz
dHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0iMjAiIHk9IjE2MCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+Tm90ZTogJ3gnIHNoYWRvd3M8L3RleHQ+CiAgICA8dGV4
dCB4PSIyMCIgeT0iMTc4IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5TY29w
ZSAxJ3MgcGFyYW1ldGVyIHghPC90ZXh0PgogIDwvZz4KPC9zdmc+
" alt="Figure 20–2: Identifier Lookup in Nested Scopes" />
<figcaption>Figure 20–2: Identifier Lookup in Nested Scopes</figcaption>
</figure>

#### 20.2.2 The Cactus-Stack (Spaghetti-Stack) Architecture

To implement lexical scoping, we organise our `Environment` as a
**Cactus Stack** (or spaghetti stack):

- Each `Environment` node contains a hash map of identifier bindings
  (`std::unordered_map<std::string, Symbol>`) and an enclosing parent
  pointer (`std::shared_ptr<Environment> parent`).
- **Insertion (`insert`):** Adds a binding exclusively to the current,
  innermost scope.
- **Lookup (`lookup`):** Searches the current scope. If found, returns
  the symbol. If not found, recursively traverses the `parent` pointer
  up the tree until either the symbol is located or the root (`nullptr`)
  is reached.

### 20.3 Type Systems: Theory and Verification

A **Type System** is a tractable syntactic method for proving the
absence of certain program behaviours by classifying phrases according
to the kinds of values they compute.

#### 20.3.1 Why Types Matter: Soundness and Milner's Dictum

In 1978, Turing Award laureate Robin Milner established the core theorem
of modern programming language theory:

$$\text{Type Safety} = \text{Progress} + \text{Preservation}$$

1.  **Progress:** A well-typed expression is never "stuck". It is either
    already a final value or can take a deterministic computational
    evaluation step.
2.  **Preservation (Subject Reduction):** If a well-typed expression $e$
    of type $\tau$ evaluates to $e'$, then $e'$ also has type $\tau$.

Together, these principles prove that a program passing our type checker
will never invoke undefined behaviour, corrupt memory layouts, or
attempt illegal machine operations at runtime.

#### 20.3.2 Bidirectional Type Checking

In compiler engineering, type checkers broadly operate in two modes (see
Figure 20–3):

1.  **Type Synthesis / Inference ($\Gamma \vdash e \Rightarrow \tau$):**
    "Given this expression $e$ and context $\Gamma$, what type does it
    produce?" (Bottom-up deduction).
2.  **Type Checking / Verification
    ($\Gamma \vdash e \Leftarrow \tau$):** "Verify that this expression
    $e$ conforms to the expected type $\tau$." (Top-down verification).

<figure id="fig:bidirectional">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MDAgMTkwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxOTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iZlNoYWRvdyIgeD0iLTUlIiB5PSItNSUi
IHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjEyMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4PSIwIiBk
eT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1jb2xvcj0idmFyKC0tc3ZnLWZnKSIgZmxvb2Qt
b3BhY2l0eT0iMC41Ii8+CiAgICA8L2ZpbHRlcj4KICA8L2RlZnM+CiAgPCEtLSBMZWZ0IEJveDog
U3ludGhlc2lzIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDYwLCAzNSkiPgogICAgPHJl
Y3QgeD0iMCIgeT0iMCIgd2lkdGg9IjM0MCIgaGVpZ2h0PSIxNTAiIHJ4PSI4IiBmaWxsPSJ2YXIo
LS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0t
c3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBmaWx0ZXI9InVybCgjZlNoYWRvdykiLz4KICAgIDx0
ZXh0IHg9IjE3MCIgeT0iMjMiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtZmFtaWx5PSJtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIj5UeXBlIFN5bnRoZXNpczogzpMg4oqiIGUg4oeSIM+EPC90ZXh0PgogICAgPHRleHQgeD0i
MjAiIHk9IjY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LWZhbWlseT0ic2Fucy1zZXJpZiIg
Zm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiPkJvdHRvbS1VcCBEZWR1Y3Rpb24gKElu
ZmVyZW5jZSk6PC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9IjkwIiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIiBmb250LWZhbWlseT0ibW9ub3NwYWNlIiBmb250LXNpemU9IjE0Ij5MaXRlcmFsOiA0MiAg
ICAgICAgICAgIOKHkiBJbnQ8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMTEwIiBmaWxsPSJ2
YXIoLS1zdmctZmcpIiBmb250LWZhbWlseT0ibW9ub3NwYWNlIiBmb250LXNpemU9IjE0Ij5MaXRl
cmFsOiAzLjE0ICAgICAgICAgIOKHkiBGbG9hdDwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSIx
MzAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtZmFtaWx5PSJtb25vc3BhY2UiIGZvbnQtc2l6
ZT0iMTQiPkJpbmFyeTogKDEwICsgMjApICAgICAg4oeSIEludDwvdGV4dD4KICA8L2c+CiAgPCEt
LSBSaWdodCBCb3g6IENoZWNraW5nIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDQ4MCwg
MzUpIj4KICAgIDxyZWN0IHg9IjAiIHk9IjAiIHdpZHRoPSIzNDAiIGhlaWdodD0iMTUwIiByeD0i
OCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBz
dHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI2ZTaGFk
b3cpIi8+CiAgICA8dGV4dCB4PSIxNzAiIHk9IjIzIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250
LWZhbWlseT0ibW9ub3NwYWNlIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgdGV4
dC1hbmNob3I9Im1pZGRsZSI+VHlwZSBWZXJpZmljYXRpb246IM6TIOKKoiBlIOKHkCDPhDwvdGV4
dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI2NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1mYW1p
bHk9InNhbnMtc2VyaWYiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIj5Ub3AtRG93
biBFeHBlY3RhdGlvbiBNYXRjaGluZzo8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iOTAiIGZp
bGw9InZhcigtLXN2Zy1mZykiIGZvbnQtZmFtaWx5PSJtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTQi
PmxldCBjb3VudDogSW50ID0gZXhwciAgIOKHkCBFeHBlY3QgSW50PC90ZXh0PgogICAgPHRleHQg
eD0iMjAiIHk9IjExMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1mYW1pbHk9Im1vbm9zcGFj
ZSIgZm9udC1zaXplPSIxNCI+aWYgKGNvbmRpdGlvbikgLi4uICAgICAg4oeQIEV4cGVjdCBCb29s
PC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9IjEzMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9u
dC1mYW1pbHk9Im1vbm9zcGFjZSIgZm9udC1zaXplPSIxNCI+Zm4gKC4uLik6IFN0cmluZyA9PiAu
Li4g4oeQIEV4cGVjdCBTdHJpbmc8L3RleHQ+CiAgPC9nPgo8L3N2Zz4=
" alt="Figure 20–3: Bottom-Up and Top-Down Type Checking" />
<figcaption>Figure 20–3: Bottom-Up and Top-Down Type
Checking</figcaption>
</figure>

### 20.4 Formal Type Judgements for Faberge

In programming language theory, type systems are specified as
mathematical inference rules using **Natural Deduction** notation:

$$\frac{\text{Premise}_1 \quad \text{Premise}_2 \quad \dots}{\text{Conclusion}}$$

Where $\Gamma$ denotes the **Type Environment** (our scoped symbol table
mapping variable identifiers $x$ to their declared types $\tau$).

#### 20.4.1 Variables and Literals

$$\frac{x : \tau \in \Gamma}{\Gamma \vdash x : \tau} \quad [\text{T-Var}] \qquad \frac{}{\Gamma \vdash n : \text{Int}} \quad [\text{T-Int}] \qquad \frac{}{\Gamma \vdash s : \text{String}} \quad [\text{T-Str}]$$

#### 20.4.2 Binary Arithmetic Operators ($+, -, *, /, \%$)

Arithmetic operators demand identical numeric operands on both sides and
synthesise the same numeric type:

$$\frac{\Gamma \vdash e_1 : \text{Int} \quad \Gamma \vdash e_2 : \text{Int}}{\Gamma \vdash e_1 + e_2 : \text{Int}} \quad [\text{T-AddInt}] \qquad \frac{\Gamma \vdash e_1 : \text{Float} \quad \Gamma \vdash e_2 : \text{Float}}{\Gamma \vdash e_1 + e_2 : \text{Float}} \quad [\text{T-AddFloat}]$$

#### 20.4.3 Comparison Operators ($==, !=, <, <=, >, >=$)

Comparisons require compatible operand types and always evaluate to
`Bool`:

$$\frac{\Gamma \vdash e_1 : \tau \quad \Gamma \vdash e_2 : \tau}{\Gamma \vdash e_1 == e_2 : \text{Bool}} \quad [\text{T-Eq}] \qquad \frac{\Gamma \vdash e_1 : \tau \quad \Gamma \vdash e_2 : \tau \quad \tau \in \{\text{Int}, \text{Float}\}}{\Gamma \vdash e_1 < e_2 : \text{Bool}} \quad [\text{T-Lt}]$$

#### 20.4.4 Conditional Expressions (`if-then-else`)

In a pure functional language, `if` is an expression that yields a
value. Thus:

1.  The condition expression must evaluate to `Bool`.
2.  Both the `then` branch and the `else` branch must evaluate to the
    **exact same type $\tau$**.

$$\frac{\Gamma \vdash e_{\text{cond}} : \text{Bool} \quad \Gamma \vdash e_{\text{then}} : \tau \quad \Gamma \vdash e_{\text{else}} : \tau}{\Gamma \vdash (\text{if } e_{\text{cond}} \text{ then } e_{\text{then}} \text{ else } e_{\text{else}}) : \tau} \quad [\text{T-If}]$$

#### 20.4.5 Lambda Functions and Function Application (Calls)

A lambda abstraction introduces its parameters into a new child
environment $\Gamma'$:

$$\frac{\Gamma, x_1 : \tau_1, \dots, x_k : \tau_k \vdash e_{\text{body}} : \tau_{\text{ret}}}{\Gamma \vdash (\text{fn } (x_1 : \tau_1, \dots, x_k : \tau_k) \Rightarrow e_{\text{body}}) : ((\tau_1, \dots, \tau_k) \to \tau_{\text{ret}})} \quad [\text{T-Lambda}]$$

A function call verifies that the callee is a function type, the
argument count matches the parameter arity, and each argument type
satisfies its corresponding parameter:

$$\frac{\Gamma \vdash e_{\text{callee}} : ((\tau_1, \dots, \tau_k) \to \tau_{\text{ret}}) \quad \Gamma \vdash a_1 : \tau_1 \quad \dots \quad \Gamma \vdash a_k : \tau_k}{\Gamma \vdash e_{\text{callee}}(a_1, \dots, a_k) : \tau_{\text{ret}}} \quad [\text{T-Call}]$$

### 20.5 Experimental Implementation in Modern C++: Semantic Analyser Architecture

#### 20.5.1 Architecture Outline

The Semantic Analyser will be architectured using clean C++20
abstractions:

- `Type`: A robust value-semantic class representing atomic types
  (`Int`, `Float`, `String`, `Bool`, `Void`), compound function
  signatures `(T1, T2 -> Ret)`, and error states.
- `SymbolTable`: A RAII-friendly lexical environment maintaining scoped
  identifier bindings.
- `TypeChecker`: An AST traversal pass implementing `ASTVisitor` that
  checks types, binds symbols, and decorates every AST node with its
  verified type.

<figure id="fig:semantic-engine">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MDguNCIgaGVp
Z2h0PSIyNjkuMiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2MDguNCAyNjkuMiI+CiAg
PHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2Jv
YiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRo
OiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1s
aW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQg
ewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9
Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3Bh
cmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3Zn
Ym9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZp
bGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFy
a2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25k
IHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRf
bWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5z
dGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2Np
cmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0
YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9j
aXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19v
cGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIg
dmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFy
a2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29u
IHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxt
YXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdv
bj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4
IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmll
bnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIy
IiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0i
b3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldp
ZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAg
ICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xl
PgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9k
ZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2MDguNCIg
aGVpZ2h0PSIyNjkuMiI+PC9yZWN0PgogIDxyZWN0IHg9IjExLjcwMDAwMSIgeT0iMTQ4LjIiIHdp
ZHRoPSIyMzQuMDAwMDIiIGhlaWdodD0iOTMuNjAwMDA2IiBjbGFzcz0ic29saWQgbm9maWxsIiBy
eD0iMCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ4Ljc1IiB5PSIx
NjcuNyIgPlNjb3BlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijk1LjU1
IiB5PSIxNjcuNyIgPlJlc29sdXRpb248L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMzMuMTUiIHk9IjE4My4zIiA+LSBwdXNoX3Njb3BlKCk8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzMuMTUiIHk9IjE5OC45MDAwMSIgPi0gcG9wX3Njb3BlKCk8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzMuMTUiIHk9IjIxNC41IiA+
LSBpbnNlcnRfc3ltYm9sKG5hbWUsIHQpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjMzLjE1IiB5PSIyMzAuMSIgPi0gbG9va3VwX3N5bWJvbChuYW1lKTwvdGV4dD4KICA8
cmVjdCB4PSIzNDcuMSIgeT0iMTQ4LjIiIHdpZHRoPSIyNDkuNiIgaGVpZ2h0PSI5My42MDAwMDYi
IGNsYXNzPSJzb2xpZCBub2ZpbGwiIHJ4PSIwIj48L3JlY3Q+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMzc2LjM1IiB5PSIxNjcuNyIgPlR5cGU8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iNDE1LjM1IiB5PSIxNjcuNyIgPkluZmVyZW5jZTwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0OTMuMzUiIHk9IjE2Ny43IiA+YW5kPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUyNC41NSIgeT0iMTY3LjciID5DaGVj
azwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNjguNTUwMDIiIHk9IjE4
My4zIiA+LSBzeW50aGVzaXNlX3R5cGUoZXhwcik8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMzY4LjU1MDAyIiB5PSIxOTguOTAwMDEiID4tIGNoZWNrX3R5cGUoZXhwciwg
ZXhwZWN0KTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNjguNTUwMDIi
IHk9IjIxNC41IiA+LSB2ZXJpZnlfY2FsbF9hcml0eSgpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjM2OC41NTAwMiIgeT0iMjMwLjEiID4tIGNoZWNrX2JyYW5jaF9lcXVh
bGl0eSgpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIxMi41NSIgeT0i
MjcuMzAwMDAxIiA+U2VtYW50aWM8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMjgyLjc1IiB5PSIyNy4zMDAwMDEiID5BbmFseXNlcjwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIzNTIuOTUiIHk9IjI3LjMwMDAwMSIgPkVuZ2luZTwvdGV4dD4KICA8
bGluZSB4MT0iMTI4LjciIHkxPSIxMDkuMjAwMDA1IiB4Mj0iMTI4LjciIHkyPSIxMjguNyIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjQ2NC4xIiB5MT0iMTA5LjIwMDAwNSIgeDI9
IjQ2NC4xIiB5Mj0iMTI4LjciIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50
cz0iMTI0LjgsMTI4LjcgMTMyLjYsMTI4LjcgMTI4LjcsMTQwLjQwMDAxIiBjbGFzcz0iZmlsbGVk
Ij48L3BvbHlnb24+CiAgPHBvbHlnb24gcG9pbnRzPSI0NjAuMiwxMjguNyA0NjgsMTI4LjcgNDY0
LjEsMTQwLjQwMDAxIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMzMuMTUiIHk9IjU4LjUiID5TeW1ib2wgVGFibGU6IHN0ZDo6c2hhcmVk
X3B0ciZsdDtFbnZpcm9ubWVudCZndDsgbV9jdXJyZW50X2VudjwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIzMy4xNSIgeT0iNzQuMSIgPlR5cGUgRGVjb3JhdG9yOiBzdGQ6
OnVub3JkZXJlZF9tYXAmbHQ7Y29uc3QgRXhwckFTVCosIFR5cGUmZ3Q7IG1fbm9kZV90eXBlczwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMy4xNSIgeT0iODkuNzAwMDA1
IiA+RGlhZ25vc3RpY3M6IERpYWdub3N0aWNFbmdpbmUgKEFOU0kgZXJyb3IgY2FyZXRzKTwvdGV4
dD4KICA8Zz4KICAgIDxsaW5lIHgxPSIxMS43MDAwMDEiIHkxPSI3LjgiIHgyPSI1OTYuNyIgeTI9
IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTEuNzAwMDAxIiB5MT0i
Ny44IiB4Mj0iMTEuNzAwMDAxIiB5Mj0iMTAxLjQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAg
PGxpbmUgeDE9IjU5Ni43IiB5MT0iNy44IiB4Mj0iNTk2LjciIHkyPSIxMDEuNCIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTEuNzAwMDAxIiB5MT0iMzkiIHgyPSI1OTYuNyIg
eTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIxMS43MDAwMDEiIHkx
PSIxMDEuNCIgeDI9IjU5Ni43IiB5Mj0iMTAxLjQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwv
Zz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 20–4: Facilities of the Semantic
Analyser</figcaption>
</figure>

#### 20.5.2 Implementation

The listings below implement the architecture described in this chapter:

- Listing 20–1 implments a strongly-typed class to hold the types
  supported by the Faberge compiler.
- Listing 20–2 creates the class which will hold named and typed symbols
  to allow for lexical scoping.
- Listing 20–3 implements class `TypeChecker`. It walks the AST,
  performs bottom-up type synthesis and top-down checking, verifies
  constraints, and records type diagnostics.
- Listing 20–4 verifies the semantic analyser against both valid
  functional programs and intentionally malformed test cases to exercise
  the compiler's diagnostic error reporting.

Listing 20–1: Strong Value-Semantic Types (Type.hpp):

In [ ]:
#ifndef TYPE_HPP
#define TYPE_HPP

#include <string>
#include <vector>
#include <memory>
#include <sstream>
#include <algorithm>

namespace Faberge {

enum class PrimitiveKind {
    Int,
    Float,
    String,
    Bool,
    Void,
    Error // Sentinel for poisonous error propagation
};

class Type {
public:
    enum class Kind { Primitive, Function };

    Type() : m_kind(Kind::Primitive), m_primitive_kind(PrimitiveKind::Void) {}
    explicit Type(PrimitiveKind pk) : m_kind(Kind::Primitive), m_primitive_kind(pk) {}

    static Type make_int()    { return Type(PrimitiveKind::Int); }
    static Type make_float()  { return Type(PrimitiveKind::Float); }
    static Type make_string() { return Type(PrimitiveKind::String); }
    static Type make_bool()   { return Type(PrimitiveKind::Bool); }
    static Type make_void()   { return Type(PrimitiveKind::Void); }
    static Type make_error()  { return Type(PrimitiveKind::Error); }

    static Type make_function(std::vector<Type> param_types, Type return_type) {
        Type t(Kind::Function);
        t.m_param_types = std::move(param_types);
        t.m_return_type = std::make_shared<Type>(std::move(return_type));
        return t;
    }

    [[nodiscard]] Kind kind() const noexcept { return m_kind; }
    [[nodiscard]] PrimitiveKind primitive_kind() const noexcept { return m_primitive_kind; }
    [[nodiscard]] bool is_error() const noexcept { 
        return m_kind == Kind::Primitive && m_primitive_kind == PrimitiveKind::Error; 
    }
    [[nodiscard]] bool is_numeric() const noexcept {
        return m_kind == Kind::Primitive && 
               (m_primitive_kind == PrimitiveKind::Int || m_primitive_kind == PrimitiveKind::Float);
    }

    [[nodiscard]] const std::vector<Type>& param_types() const noexcept { return m_param_types; }
    [[nodiscard]] const Type& return_type() const { return *m_return_type; }

    // Strict structural type equality
    bool operator==(const Type& other) const {
        if (is_error() || other.is_error()) return true; // Suppress cascading error spam
        if (m_kind != other.m_kind) return false;

        if (m_kind == Kind::Primitive) {
            return m_primitive_kind == other.m_primitive_kind;
        }

        if (m_param_types.size() != other.m_param_types.size()) return false;
        for (size_t i = 0; i < m_param_types.size(); ++i) {
            if (m_param_types[i] != other.m_param_types[i]) return false;
        }
        return *m_return_type == *other.m_return_type;
    }

    bool operator!=(const Type& other) const { return !(*this == other); }

    [[nodiscard]] std::string to_string() const {
        if (m_kind == Kind::Primitive) {
            switch (m_primitive_kind) {
                case PrimitiveKind::Int:    return "Int";
                case PrimitiveKind::Float:  return "Float";
                case PrimitiveKind::String: return "String";
                case PrimitiveKind::Bool:   return "Bool";
                case PrimitiveKind::Void:   return "Void";
                case PrimitiveKind::Error:  return "<type-error>";
            }
        }

        std::ostringstream ss;
        ss << "(";
        for (size_t i = 0; i < m_param_types.size(); ++i) {
            ss << m_param_types[i].to_string();
            if (i + 1 < m_param_types.size()) ss << ", ";
        }
        ss << " -> " << m_return_type->to_string() << ")";
        return ss.str();
    }

private:
    explicit Type(Kind k) : m_kind(k) {}
    Kind m_kind;
    PrimitiveKind m_primitive_kind{PrimitiveKind::Void};
    std::vector<Type> m_param_types;
    std::shared_ptr<Type> m_return_type;
};

} // namespace Faberge

#endif // TYPE_HPP

Listing 20–2: Scoped Environment Tree (SymbolTable.hpp):

In [ ]:
#ifndef SYMBOL_TABLE_HPP
#define SYMBOL_TABLE_HPP

#include "faberge-headers/Type.hpp"
#include "faberge-headers/SourceLocation.hpp"
#include <string>
#include <unordered_map>
#include <memory>
#include <optional>
#include <stdexcept>

namespace Faberge {

struct Symbol {
    std::string name;
    Type type;
    SourceLocation location;
    bool is_function{false};
};

class Environment : public std::enable_shared_from_this<Environment> {
public:
    explicit Environment(std::shared_ptr<Environment> parent = nullptr)
        : m_parent(std::move(parent)) {}

    static std::shared_ptr<Environment> create(std::shared_ptr<Environment> parent = nullptr) {
        return std::make_shared<Environment>(std::move(parent));
    }

    [[nodiscard]] std::shared_ptr<Environment> parent() const noexcept { return m_parent; }

    bool define(std::string name, Type type, SourceLocation loc, bool is_fn = false) {
        if (m_bindings.contains(name)) {
            return false; // Duplicate symbol declaration in the EXACT same scope
        }
        m_bindings.emplace(name, Symbol{name, std::move(type), std::move(loc), is_fn});
        return true;
    }

    [[nodiscard]] std::optional<Symbol> lookup(const std::string& name) const {
        auto it = m_bindings.find(name);
        if (it != m_bindings.end()) {
            return it->second;
        }
        if (m_parent) {
            return m_parent->lookup(name);
        }
        return std::nullopt;
    }

private:
    std::shared_ptr<Environment> m_parent;
    std::unordered_map<std::string, Symbol> m_bindings;
};

} // namespace Faberge

#endif // SYMBOL_TABLE_HPP

Listing 20–3: Type Checker Engine (TypeChecker.hpp):

In [ ]:
#ifndef TYPE_CHECKER_HPP
#define TYPE_CHECKER_HPP

#include "faberge-headers/AST.hpp"
#include "faberge-headers/ASTVisitor.hpp"
#include "faberge-headers/SymbolTable.hpp"
#include "faberge-headers/Diagnostic.hpp"
#include <unordered_map>

namespace Faberge {

class TypeChecker : public ASTVisitor {
public:
    explicit TypeChecker(std::string_view source_code)
        : m_source_code(source_code), 
          m_current_env(Environment::create(nullptr)) {}

    bool check(const ProgramAST& program) {
        m_error_count = 0;
        program.accept(*this);
        return m_error_count == 0;
    }

    [[nodiscard]] size_t error_count() const noexcept { return m_error_count; }

    [[nodiscard]] Type get_node_type(const ExprAST& expr) const {
        auto it = m_node_types.find(&expr);
        if (it != m_node_types.end()) return it->second;
        return Type::make_error();
    }

    // -------------------------------------------------------------------------
    // Visitor Dispatch Implementations
    // -------------------------------------------------------------------------

    void visit(const ProgramAST& node) override {
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
    }

    void visit(const LetStmtAST& node) override {
        // Pre-bind function identifier in environment to support recursive self-references
        const auto* lambda = dynamic_cast<const LambdaExprAST*>(&node.initialiser());
        bool prebound = false;
        Type prebound_type = Type::make_error();

        if (node.type_annotation().has_value()) {
            prebound_type = convert_ast_type(*node.type_annotation());
            prebound = true;
        } else if (lambda && lambda->return_type().has_value()) {
            std::vector<Type> param_types;
            for (const auto& p : lambda->parameters()) {
                param_types.push_back(convert_ast_type(p.type));
            }
            Type ret_type = convert_ast_type(*lambda->return_type());
            prebound_type = Type::make_function(std::move(param_types), ret_type);
            prebound = true;
        }

        if (prebound) {
            if (!m_current_env->define(node.name(), prebound_type, node.location(), lambda != nullptr)) {
                report_type_error(node.location(), 
                                  "Redeclaration of identifier '" + node.name() + "' in the same scope.");
            }
        }

        node.initialiser().accept(*this);
        Type init_type = get_node_type(node.initialiser());

        Type var_type = init_type;

        // If explicit type annotation exists, verify compatibility
        if (node.type_annotation().has_value()) {
            Type declared_type = convert_ast_type(*node.type_annotation());
            if (init_type != declared_type && !init_type.is_error()) {
                report_type_error(node.initialiser().location(),
                                  "Type mismatch in variable declaration: expected '" + 
                                  declared_type.to_string() + "', got '" + 
                                  init_type.to_string() + "'.");
            }
            var_type = declared_type;
        }

        if (!prebound) {
            if (!m_current_env->define(node.name(), var_type, node.location())) {
                report_type_error(node.location(), 
                                  "Redeclaration of identifier '" + node.name() + "' in the same scope.");
            }
        }
    }

    void visit(const ExprStmtAST& node) override {
        node.expression().accept(*this);
    }

    void visit(const LiteralExprAST& node) override {
        Type t = Type::make_error();
        switch (node.token_type()) {
            case TokenType::Literal_Integer: t = Type::make_int(); break;
            case TokenType::Literal_Float:   t = Type::make_float(); break;
            case TokenType::Literal_String:  t = Type::make_string(); break;
            case TokenType::Literal_Boolean: t = Type::make_bool(); break;
            default: break;
        }
        set_node_type(node, t);
    }

    void visit(const VariableExprAST& node) override {
        auto sym = m_current_env->lookup(node.name());
        if (sym == std::nullopt) {
            report_type_error(node.location(), 
                              "Use of undeclared identifier '" + node.name() + "'.");
            set_node_type(node, Type::make_error());
            return;
        }
        set_node_type(node, sym->type);
    }

    void visit(const UnaryExprAST& node) override {
        node.operand().accept(*this);
        Type operand_type = get_node_type(node.operand());

        if (node.op() == TokenType::Minus) {
            if (!operand_type.is_numeric() && !operand_type.is_error()) {
                report_type_error(node.location(), 
                                  "Unary '-' operator requires numeric operand, found '" + 
                                  operand_type.to_string() + "'.");
                set_node_type(node, Type::make_error());
                return;
            }
            set_node_type(node, operand_type);
            return;
        }

        if (node.op() == TokenType::Exclamation) {
            if (operand_type != Type::make_bool() && !operand_type.is_error()) {
                report_type_error(node.location(), 
                                  "Logical '!' operator requires Bool operand, found '" + 
                                  operand_type.to_string() + "'.");
                set_node_type(node, Type::make_error());
                return;
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        set_node_type(node, Type::make_error());
    }

    void visit(const BinaryExprAST& node) override {
        node.left().accept(*this);
        node.right().accept(*this);

        Type left_t = get_node_type(node.left());
        Type right_t = get_node_type(node.right());

        if (left_t.is_error() || right_t.is_error()) {
            set_node_type(node, Type::make_error());
            return;
        }

        // 1. Arithmetic Operators (+, -, *, /, %)
        if (node.op() == TokenType::Plus || node.op() == TokenType::Minus ||
            node.op() == TokenType::Asterisk || node.op() == TokenType::Slash || 
            node.op() == TokenType::Percent) {
            
            // Allow string concatenation on '+'
            if (node.op() == TokenType::Plus && left_t == Type::make_string() && right_t == Type::make_string()) {
                set_node_type(node, Type::make_string());
                return;
            }

            if (!left_t.is_numeric() || !right_t.is_numeric() || left_t != right_t) {
                report_type_error(node.location(), 
                                  "Binary operator '" + std::string(Token::to_string(node.op())) + 
                                  "' expects matching numeric operands, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
                set_node_type(node, Type::make_error());
                return;
            }

            set_node_type(node, left_t);
            return;
        }

        // 2. Comparison Operators (==, !=, <, <=, >, >=)
        if (node.op() == TokenType::EqualEqual || node.op() == TokenType::ExclamationEqual) {
            if (left_t != right_t) {
                report_type_error(node.location(), 
                                  "Equality operator requires identical operand types, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        if (node.op() == TokenType::Less || node.op() == TokenType::LessEqual ||
            node.op() == TokenType::Greater || node.op() == TokenType::GreaterEqual) {
            if (!left_t.is_numeric() || !right_t.is_numeric() || left_t != right_t) {
                report_type_error(node.location(), 
                                  "Relational comparison requires matching numeric types, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        // 3. Logical Operators (&&, ||)
        if (node.op() == TokenType::AmpersandAmpersand || node.op() == TokenType::PipePipe) {
            if (left_t != Type::make_bool() || right_t != Type::make_bool()) {
                report_type_error(node.location(), 
                                  "Logical operator requires Bool operands, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        set_node_type(node, Type::make_error());
    }

    void visit(const IfExprAST& node) override {
        node.condition().accept(*this);
        Type cond_t = get_node_type(node.condition());

        if (cond_t != Type::make_bool() && !cond_t.is_error()) {
            report_type_error(node.condition().location(), 
                              "If condition must evaluate to Bool, found '" + cond_t.to_string() + "'.");
        }

        node.then_branch().accept(*this);
        node.else_branch().accept(*this);

        Type then_t = get_node_type(node.then_branch());
        Type else_t = get_node_type(node.else_branch());

        if (then_t != else_t && !then_t.is_error() && !else_t.is_error()) {
            report_type_error(node.location(), 
                              "Branches of 'if' expression must have matching types. Then branch is '" + 
                              then_t.to_string() + "', Else branch is '" + else_t.to_string() + "'.");
            set_node_type(node, Type::make_error());
            return;
        }

        set_node_type(node, then_t);
    }

    void visit(const CallExprAST& node) override {
        node.callee().accept(*this);
        Type callee_t = get_node_type(node.callee());

        if (callee_t.is_error()) {
            set_node_type(node, Type::make_error());
            return;
        }

        if (callee_t.kind() != Type::Kind::Function) {
            report_type_error(node.location(), 
                              "Attempted to call non-function expression of type '" + 
                              callee_t.to_string() + "'.");
            set_node_type(node, Type::make_error());
            return;
        }

        if (node.arguments().size() != callee_t.param_types().size()) {
            report_type_error(node.location(), 
                              "Function expects " + std::to_string(callee_t.param_types().size()) + 
                              " arguments, but received " + std::to_string(node.arguments().size()) + ".");
            set_node_type(node, callee_t.return_type());
            return;
        }

        for (size_t i = 0; i < node.arguments().size(); ++i) {
            node.arguments()[i]->accept(*this);
            Type arg_t = get_node_type(*node.arguments()[i]);
            Type expected_param_t = callee_t.param_types()[i];

            if (arg_t != expected_param_t && !arg_t.is_error()) {
                report_type_error(node.arguments()[i]->location(), 
                                  "Argument " + std::to_string(i + 1) + " type mismatch: expected '" + 
                                  expected_param_t.to_string() + "', got '" + arg_t.to_string() + "'.");
            }
        }

        set_node_type(node, callee_t.return_type());
    }

    void visit(const BlockExprAST& node) override {
        push_scope();

        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }

        Type block_result_t = Type::make_void();
        if (node.result_expr()) {
            node.result_expr()->accept(*this);
            block_result_t = get_node_type(*node.result_expr());
        }

        pop_scope();
        set_node_type(node, block_result_t);
    }

    void visit(const LambdaExprAST& node) override {
        std::vector<Type> param_types;
        
        // Construct child environment for function parameters
        auto fn_env = Environment::create(m_current_env);

        for (const auto& p : node.parameters()) {
            Type pt = convert_ast_type(p.type);
            param_types.push_back(pt);
            fn_env->define(p.name, pt, p.location);
        }

        // Switch to function environment to evaluate body
        auto prev_env = m_current_env;
        m_current_env = fn_env;

        node.body().accept(*this);
        Type body_t = get_node_type(node.body());

        m_current_env = prev_env;

        Type return_t = body_t;
        if (node.return_type().has_value()) {
            Type declared_ret = convert_ast_type(*node.return_type());
            if (body_t != declared_ret && !body_t.is_error()) {
                report_type_error(node.body().location(), 
                                  "Function body type mismatch: expected return type '" + 
                                  declared_ret.to_string() + "', but body evaluates to '" + 
                                  body_t.to_string() + "'.");
            }
            return_t = declared_ret;
        }

        Type fn_type = Type::make_function(std::move(param_types), return_t);
        set_node_type(node, fn_type);
    }

private:
    std::string_view m_source_code;
    std::shared_ptr<Environment> m_current_env;
    std::unordered_map<const ExprAST*, Type> m_node_types;
    size_t m_error_count{0};

    void push_scope() {
        m_current_env = Environment::create(m_current_env);
    }

    void pop_scope() {
        if (m_current_env->parent()) {
            m_current_env = m_current_env->parent();
        }
    }

    void set_node_type(const ExprAST& expr, Type t) {
        m_node_types[&expr] = std::move(t);
    }

    void report_type_error(SourceLocation loc, std::string_view msg) {
        ++m_error_count;
        DiagnosticEngine::report_error(m_source_code, loc, msg);
    }

    static Type convert_ast_type(const TypeNode& ast_type) {
        switch (ast_type.kind()) {
            case TypeNode::Kind::Primitive:
                switch (ast_type.primitive_kind()) {
                    case PrimitiveTypeKind::Int:    return Type::make_int();
                    case PrimitiveTypeKind::Float:  return Type::make_float();
                    case PrimitiveTypeKind::String: return Type::make_string();
                    case PrimitiveTypeKind::Bool:   return Type::make_bool();
                    case PrimitiveTypeKind::Void:   return Type::make_void();
                }
                break;
            case TypeNode::Kind::Function:
                // Extendable for higher-order type annotations
                return Type::make_error();
            case TypeNode::Kind::List:
                return Type::make_error();
        }
        return Type::make_error();
    }
};

} // namespace Faberge

#endif // TYPE_CHECKER_HPP

Listing 20–4: Verification and Negative Diagnostics Showcase:

In [ ]:
#include "faberge-headers/Lexer.hpp"
#include "faberge-headers/Parser.hpp"
#include "faberge-headers/TypeChecker.hpp"
#include <iostream>

void run_semantic_test(std::string_view title, std::string_view source_code) {
    using namespace Faberge;

    std::cout << "\n====================================================\n";
    std::cout << " TEST CASE: " << title << "\n";
    std::cout << "====================================================\n";
    std::cout << "SOURCE CODE:\n" << source_code << "\n----------------------------------------------------\n";

    // Stage 1: Lexical Analysis
    Lexer lexer(source_code, "test.fab");
    auto tokens = lexer.tokenise();
    if (lexer.has_errors()) {
        std::cerr << "[Abort]: Lexer errors encountered.\n";
        return;
    }

    // Stage 2: Syntactic Analysis (Parser)
    Parser parser(std::move(tokens), source_code);
    auto ast = parser.parse_program();
    if (parser.has_errors()) {
        std::cerr << "[Abort]: Parser errors encountered.\n";
        return;
    }

    // Stage 3: Semantic Analysis and Type Checking
    TypeChecker checker(source_code);
    bool passed = checker.check(*ast);

    if (passed) {
        std::cout << "\033[1;32m[PASS]: Static Semantic and Type Verification Succeeded (0 Errors).\033[0m\n";
    } else {
        std::cout << "\033[1;31m[FAIL]: Semantic Analysis Failed with " 
                  << checker.error_count() << " error(s).\033[0m\n";
    }
}

int main() {
    // 1. Valid Higher-Order Recursive Program
    constexpr std::string_view valid_source = R"(
        let add = fn (a: Int, b: Int): Int => a + b;

        let factorial = fn (n: Int): Int => {
            if n <= 1 then 1 else n * factorial(n - 1)
        };

        let main = fn (): Int => {
            let x = 10;
            let result = factorial(x);
            add(result, 42)
        };
    )";
    run_semantic_test("Valid Recursive Functional Program", valid_source);

    // 2. Intentional Semantic Violations
    constexpr std::string_view invalid_source = R"(
        let bad_math = 10 + "hello"; // Error 1: Type mismatch in addition

        let bad_branch = if true then 42 else "divergent string"; // Error 2: Branch type mismatch

        let bad_call = fn (x: Int): Int => {
            let local_flag = true;
            undeclared_function(x, local_flag) // Error 3: Undeclared identifier
        };

        let return_mismatch = fn (a: Int): Int => {
            "this is not an integer" // Error 4: Return type mismatch
        };
    )";
    //run_semantic_test("Negative Semantic Tests (Intentional Errors)", invalid_source);

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

### 20.6 Summary

In this third stage of our compiler project, we transitioned from
structural syntax to semantic meaning and type safety:

- **Semantic Analysis** enforces logical invariants that context-free
  grammars cannot express, verifying identifier existence, visibility,
  and operand type compatibility.
- **The Scoped Symbol Table (Cactus Stack)** mirrors lexical scoping
  hierarchies, supporting nested child environments and variable
  shadowing.
- **Milner's Dictum** proves that a sound static type system prevents
  well-typed programs from entering undefined or illegal runtime states.
- **Bidirectional Type Checking** combines bottom-up synthesis with
  top-down expected-type checking to deduce and verify expression types
  cleanly.

### 20.7 Programming Challenges

#### Exercise 20.1: Immutable vs Mutable Binding Enforcement (`let` vs `mut`)

**Context and Theory:**

In modern systems and functional languages, immutability is the default
state of values. In Faberge, a binding declared via `let` is immutable:
once assigned an initial value, its memory cannot be reassigned. To
allow mutation, the developer must explicitly annotate the binding with
the `mut` keyword:

$$\text{let } x = 10; \quad x = 20; \quad \implies \text{Semantic Error (Mutation of Immutable Binding)}$$

$$\text{let mut } y = 10; \quad y = 20; \quad \implies \text{Valid (Mutable Binding)}$$

The semantic analyser enforces two invariants during assignment
expressions ($x = e$):

1.  **Mutability Invariant:** The target symbol $x$ must be declared
    with `is_mutable = true` in the active scope environment $\Gamma$.
2.  **Type Invariance:** The type of the assigned expression $e$ must
    match the declared type of $x$:

$$\frac{x : \tau \in \Gamma \quad \text{is\_mutable}(x) = \text{true} \quad \Gamma \vdash e : \tau}{\Gamma \vdash (x = e) : \tau} \quad [\text{T-Assign}]$$

<figure id="fig:mutable">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2ODYuNCIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDY4Ni40IDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNjg2LjQiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUi
IHk9IjExLjcwMDAwMSIgPls8L3RleHQ+CiAgPGxpbmUgeDE9IjM5MCIgeTE9IjcuOCIgeDI9IjQx
My40MDAwMiIgeTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRz
PSI0MTMuNDAwMDIsMy45IDQyMS4yLDcuOCA0MTMuNDAwMDIsMTEuNzAwMDAxIiBjbGFzcz0iZmls
bGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDMwLjk1IiB5
PSIxMS43MDAwMDEiID5NdXRhdGlvbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI1MDEuMTUwMDIiIHk9IjExLjcwMDAwMSIgPkF0dGVtcHQ8L3RleHQ+CiAgPGxpbmUgeDE9
IjU2MS42MDAwNCIgeTE9IjcuOCIgeDI9IjU4NSIgeTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgPHBvbHlnb24gcG9pbnRzPSI1ODUsMy45IDU5Mi44LDcuOCA1ODUsMTEuNzAwMDAxIiBj
bGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NjAyLjU1IiB5PSIxMS43MDAwMDEiID5UeXBlRXJyb3IhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjQyLjkiID5bPC90ZXh0PgogIDxsaW5lIHgxPSIzOTAi
IHkxPSIzOSIgeDI9IjQxMy40MDAwMiIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8
cG9seWdvbiBwb2ludHM9IjQxMy40MDAwMiwzNS4xMDAwMDIgNDIxLjIsMzkgNDEzLjQwMDAyLDQy
LjkiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI0MzAuOTUiIHk9IjQyLjkiID5NdXRhdGlvbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI1MDEuMTUwMDIiIHk9IjQyLjkiID5BdHRlbXB0PC90ZXh0PgogIDxsaW5l
IHgxPSI1NjEuNjAwMDQiIHkxPSIzOSIgeDI9IjU4NSIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICA8cG9seWdvbiBwb2ludHM9IjU4NSwzNS4xMDAwMDIgNTkyLjgsMzkgNTg1LDQyLjki
IGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI2MDIuNTUiIHk9IjQyLjkiID5PSzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyNS4zNSIgeT0iMTEuNzAwMDAxIiA+U3ltYm9sOiAmIzM5O3gmIzM5OyB8IFR5cGU6IElu
dCB8IGlzX211dGFibGU6IGZhbHNlIF08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMjUuMzUiIHk9IjQyLjkiID5TeW1ib2w6ICYjMzk7eSYjMzk7IHwgVHlwZTogSW50IHwg
aXNfbXV0YWJsZTogdHJ1ZSAgXTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 20–5: Symbol Table Entry</figcaption>
</figure>

**Specification:**

Extend the `Symbol` descriptor with a `bool is_mutable` flag. Implement
a new AST node
`AssignExprAST(SourceLocation, std::string name, std::unique_ptr<ExprAST> value)`.
In `TypeChecker`, implement the assignment visitor: verify that the
identifier exists, assert that it is mutable, and check that the
assigned expression matches the variable's type.

``` cpp
struct Symbol {
    std::string name;
    Type type;
    SourceLocation location;
    bool is_mutable{false};
};

class TypeCheckerExtension {
public:
    void check_assignment(const std::string& name, const ExprAST& val_expr, SourceLocation loc) {
        auto sym = m_current_env->lookup(name);
        if (!sym) {
            report_error(loc, "Assignment to undeclared identifier '" + name + "'.");
            return;
        }
        if (!sym->is_mutable) {
            report_error(loc, "Cannot reassign to immutable binding '" + name + 
                              "'. Declared with 'let'; did you mean 'let mut'?");
        }
        Type val_type = synthesise_type(val_expr);
        if (val_type != sym->type && !val_type.is_error()) {
            report_error(loc, "Type mismatch in assignment to '" + name + "': expected '" + 
                              sym->type.to_string() + "', got '" + val_type.to_string() + "'.");
        }
    }
};
```

**Verification Tasks:**

1.  **Immutable Reassignment Rejection:** Attempt to reassign an
    immutable variable: `let x = 10; x = 20;`. Verify that the type
    checker rejects the program with an immutability violation error.
2.  **Valid Mutable Mutation:** Check
    `let mut count = 0; count = count + 1;`. Verify that the assignment
    passes with zero errors and synthesises type `Int`.
3.  **Mutable Type Mismatch:** Attempt to assign a string to a mutable
    integer: `let mut y = 42; y = "forty-two";`. Verify that the
    analyser catches the type mismatch and preserves the variable's
    original integer type.

#### Exercise 20.2: First-Class Homogeneous List and Subscript Type Verification

**Context and Theory:**

In Faberge, sequential collections are represented as **Homogeneous
Lists** $[e_1, e_2, \dots, e_k]$, where every element shares an
identical type $\tau$. The collection itself has the compound parametric
type $\text{List}\langle\tau\rangle$.

The type checker enforces two formal deduction rules (see Figure 20–6):

1.  **List Construction:** Every element expression in the literal must
    unify with a common type $\tau$. If the list is empty `[]`, its type
    must be inferred from surrounding contextual expectations (e.g. an
    explicit type annotation):

$$\frac{\Gamma \vdash e_1 : \tau \quad \Gamma \vdash e_2 : \tau \quad \dots \quad \Gamma \vdash e_k : \tau}{\Gamma \vdash [e_1, e_2, \dots, e_k] : \text{List}\langle\tau\rangle} \quad [\text{T-List}]$$

1.  **Subscript Indexing:** The collection target must evaluate to
    $\text{List}\langle\tau\rangle$, and the subscript index must
    evaluate strictly to `Int`:

$$\frac{\Gamma \vdash e_{\text{target}} : \text{List}\langle\tau\rangle \quad \Gamma \vdash e_{\text{index}} : \text{Int}}{\Gamma \vdash e_{\text{target}}[e_{\text{index}}] : \tau} \quad [\text{T-Index}]$$

<figure id="fig:type-checker">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MjQiIGhlaWdo
dD0iOTcuNjAwMDA2IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYyNCA5Ny42MDAwMDYi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNw
eDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJh
bnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoK
LnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmls
bCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlh
bW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFy
a2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0K
Ci5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Bl
bl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtl
ci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29w
ZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNi
aWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJy
b3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9Ijci
IG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9s
eWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAg
ICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9
IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1y
ZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3Bv
bHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2Np
cmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3
Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQi
IGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgog
IDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNjI0
IiBoZWlnaHQ9Ijk3LjYwMDAwNiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjkuNzUiIHk9IjExLjcwMDAwMSIgPkxpdGVyYWw6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9Ijc5Ljk1MDAwNSIgeT0iMTEuNzAwMDAxIiA+WzwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5NS41NSIgeT0iMTEuNzAwMDAxIiA+MTAsPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEyNi43NSIgeT0iMTEuNzAwMDAxIiA+
MjAsPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE1Ny45NSIgeT0iMTEu
NzAwMDAxIiA+MzA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTgxLjM1
IiB5PSIxMS43MDAwMDEiID5dPC90ZXh0PgogIDxsaW5lIHgxPSIyNzMiIHkxPSI3LjgiIHgyPSIy
OTYuNCIgeTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIy
OTYuNCwzLjkgMzA0LjIsNy44IDI5Ni40LDExLjcwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5
Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMxMy45NSIgeT0iMTEuNzAwMDAx
IiA+SW5mZXJyZWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzg0LjE1
MDAyIiB5PSIxMS43MDAwMDEiID5UeXBlOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI0MzAuOTUiIHk9IjExLjcwMDAwMSIgPkxpc3QmbHQ7SW50Jmd0OzwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5PSI0Mi45IiA+SW5kZXg6PC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijc5Ljk1MDAwNSIgeT0iNDIuOSIgPm51
bWJlcnNbMF08L3RleHQ+CiAgPGxpbmUgeDE9IjI3MyIgeTE9IjM5IiB4Mj0iMjk2LjQiIHkyPSIz
OSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyOTYuNCwzNS4xMDAw
MDIgMzA0LjIsMzkgMjk2LjQsNDIuOSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMxMy45NSIgeT0iNDIuOSIgPlN5bnRoZXNpc2VkPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQwNy41NTAwMiIgeT0iNDIuOSIg
PlR5cGU6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ1NC4zNSIgeT0i
NDIuOSIgPkludDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5
PSI3NC4xIiA+SWxsZWdhbDo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NzkuOTUwMDA1IiB5PSI3NC4xIiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI5NS41NSIgeT0iNzQuMSIgPjEwLDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxMjYuNzUiIHk9Ijc0LjEiID4mIzM5O3N0cmluZyYjMzk7LDwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMDQuNzUiIHk9Ijc0LjEiID50cnVlPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0My43NSIgeT0iNzQuMSIgPl08L3RleHQ+
CiAgPGxpbmUgeDE9IjI3MyIgeTE9IjcwLjIwMDAwNSIgeDI9IjI5Ni40IiB5Mj0iNzAuMjAwMDA1
IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjI5Ni40LDY2LjMgMzA0
LjIsNzAuMjAwMDA1IDI5Ni40LDc0LjEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMTMuOTUiIHk9Ijc0LjEiID5TZW1hbnRpYzwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzODQuMTUwMDIiIHk9Ijc0LjEiID5F
cnJvcjo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDM4Ljc1IiB5PSI3
NC4xIiA+SGV0ZXJvZ2VuZW91czwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI1NDcuOTUiIHk9Ijc0LjEiID5FbGVtZW50cyE8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 20–6: List and Subscript Type
Verification</figcaption>
</figure>

**Specification:**

Extend Faberge's `Type` class with `Kind::List` containing an inner
element `std::shared_ptr<Type>`. Implement visitor methods in
`TypeChecker` for `ListLiteralExprAST` and `IndexExprAST`, enforcing
element type homogeneity and integer-only indexing.

``` cpp
class ListTypeChecker {
public:
    Type check_list_literal(const ListLiteralExprAST& node) {
        if (node.elements().empty()) return Type::make_list(Type::make_unknown());
        Type element_type = synthesise_type(*node.elements()[0]);
        for (size_t i = 1; i < node.elements().size(); ++i) {
            Type t = synthesise_type(*node.elements()[i]);
            if (t != element_type && !t.is_error()) {
                report_error(node.elements()[i]->location(), 
                             "List element type mismatch: expected '" + element_type.to_string() + 
                             "', got '" + t.to_string() + "'.");
            }
        }
        return Type::make_list(element_type);
    }
};
```

**Verification Tasks:**

1.  **Homogeneous List Deduction:** Check `let primes = [2, 3, 5, 7];`.
    Verify that the variable `primes` is entered into the symbol table
    with type `List<Int>`.
2.  **Heterogeneous List Rejection:** Check
    `let mixed = [1, "two", 3];`. Verify that the type checker flags the
    second element as a type mismatch.
3.  **Non-Integer Index Rejection:** Check
    `let arr = [10, 20]; let v = arr["first"];`. Verify that the
    analyser rejects the string index, stating that subscript operations
    demand an `Int`.

#### Exercise 20.3: Heterogeneous Product Types (Tuples) and Arity Equivalence

**Context and Theory:**

In contrast to homogeneous lists, a **Tuple** is a heterogeneous product
type whose components can each possess a distinct type:

$$\tau_{\text{tuple}} = (\tau_1 \times \tau_2 \times \dots \times \tau_k)$$

Tuple types obey **Structural Equivalence**. Two tuple types are
identical if and only if their arities match and their component types
are pairwise identical:

$$(\tau_1, \dots, \tau_n) \equiv (\sigma_1, \dots, \sigma_m) \iff n = m \land \forall i \in [1, n]: \tau_i \equiv \sigma_i$$

Tuple component extraction using dot-indexing (e.g. `pair.0`, `pair.1`)
must be checked statically against the tuple's compile-time arity $k$.
Attempting to access an index out of bounds ($\text{idx} \ge k$) is an
illegal semantic error (see Figure 20–7).

<figure id="fig:tuple-equivalence">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI3NTYuNjAwMDQi
IGhlaWdodD0iOTcuNjAwMDA2IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDc1Ni42MDAw
NCA5Ny42MDAwMDYiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2Jv
YiBjaXJjbGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNr
OwogIHN0cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0
eTogMTsKICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsK
fQoKLnN2Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZv
bnQtZmFtaWx5OiBDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZv
bnQtc2l6ZTogMTNweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hh
cnJheTogODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAu
YmdfZmlsbGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5z
dmdib2IgLm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfYXJyb3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfYXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVu
ZF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1l
bmQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJj
bGUgewogIG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9t
YXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNs
ZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1h
cmtlciBpZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFy
a2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2Ui
PgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAy
LDQgMCwyIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiBy
ZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdf
ZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5f
Y2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxj
aXJjbGUgY3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAg
IDwvbWFya2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00
IiB3aWR0aD0iNzU2LjYwMDA0IiBoZWlnaHQ9Ijk3LjYwMDAwNiI+PC9yZWN0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjE1Ny45NSIgeT0iMTEuNzAwMDAxIiA+RmFiZXJnZTwvdGV4
dD4KICA8bGluZSB4MT0iMjg4LjYiIHkxPSI3LjgiIHgyPSIzMTIiIHkyPSI3LjgiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzEyLDMuOSAzMTkuODAwMDIsNy44IDMx
MiwxMS43MDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iMjg4LjYi
IHkxPSIzOSIgeDI9IjMxMiIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdv
biBwb2ludHM9IjMxMiwzNS4xMDAwMDIgMzE5LjgwMDAyLDM5IDMxMiw0Mi45IiBjbGFzcz0iZmls
bGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjI4OC42IiB5MT0iNzAuMjAwMDA1IiB4Mj0iMzEy
IiB5Mj0iNzAuMjAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9
IjMxMiw2Ni4zIDMxOS44MDAwMiw3MC4yMDAwMDUgMzEyLDc0LjEiIGNsYXNzPSJmaWxsZWQiPjwv
cG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5PSIxMS43MDAw
MDEiID5FeHByZXNzaW9uOiAoNDIsIDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyMTIuNTUiIHk9IjExLjcwMDAwMSIgPiwgdHJ1ZSk8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMzI5LjU1MDAyIiB5PSIxMS43MDAwMDEiID5UeXBlOiAoSW50ICog
U3RyaW5nICogQm9vbCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43
NSIgeT0iNDIuOSIgPkFjY2VzczogICAgIHQuMDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIzMjkuNTUwMDIiIHk9IjQyLjkiID5UeXBlOiBJbnQ8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0iNzQuMSIgPkFjY2VzczogICAgIHQuMzwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjkuNTUwMDIiIHk9Ijc0LjEi
ID5Db21waWxlLVRpbWUgRXJyb3I6IEluZGV4IDMgb3V0IG9mIGJvdW5kcyBmb3IgYXJpdHkgMyE8
L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 20–7: Structural Equivalence in Tuples</figcaption>
</figure>

**Specification:**

Extend `Type` with `Kind::Tuple` holding `std::vector<Type>`. Implement
type checking for `TupleExprAST` and a member-access expression
`TupleAccessExprAST(SourceLocation, std::unique_ptr<ExprAST> target, size_t index)`.

``` cpp
class TupleTypeChecker {
public:
    Type check_tuple_access(const ExprAST& target, size_t index, SourceLocation loc) {
        Type target_t = synthesise_type(target);
        if (target_t.kind() != Type::Kind::Tuple) {
            report_error(loc, "Cannot perform tuple indexing on non-tuple type '" + 
                              target_t.to_string() + "'.");
            return Type::make_error();
        }
        if (index >= target_t.tuple_element_types().size()) {
            report_error(loc, "Tuple index " + std::to_string(index) + " out of bounds for tuple of arity " + 
                              std::to_string(target_t.tuple_element_types().size()) + ".");
            return Type::make_error();
        }
        return target_t.tuple_element_types()[index];
    }
};
```

**Verification Tasks:**

1.  **Tuple Synthesis:** Check `let record = (101, "Alice", 3.14);`.
    Verify that `record` synthesises type `(Int, String, Float)`.
2.  **Static Index Projection:** Check `let name = (101, "Alice").1;`.
    Verify that `name` is statically verified as type `String`.
3.  **Out-of-Bounds Rejection:** Check `let invalid = (10, 20).5;`.
    Verify that the checker flags an index out-of-bounds error at
    compile time.

#### Exercise 20.4: Recursive Function Self-Binding and Mutual Recursion

**Context and Theory:**

In a functional language, functions can reference themselves
recursively:

```
let factorial = fn (n: Int): Int => {
    if n <= 1 then 1 else n * factorial(n - 1)
};
```

Under a naive sequential symbol table, evaluating the lambda body fails
with an `Undeclared Identifier` error because `factorial` has not yet
been bound in the enclosing environment!

Furthermore, **Mutual Recursion** occurs when function $f$ invokes $g$,
and $g$ invokes $f$:

```
let is_even = fn (n: Int): Bool => if n == 0 then true else is_odd(n - 1);
let is_odd  = fn (n: Int): Bool => if n == 0 then false else is_even(n - 1);
```

To resolve this, the semantic analyser must execute a **Two-Pass
Environment Setup** over program blocks (see Figure 20–8):

- **Pass 1 (Hoisting / Signature Registration):** Traverse all top-level
  `LetStmtAST` nodes. If a statement binds a function with explicit
  parameter and return types, pre-declare the function symbol in the
  environment *before* checking function bodies.
- **Pass 2 (Body Verification):** Traverse the function bodies. All
  function names are now mutually visible.

<figure id="fig:two-pass-environment">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NjAuMiIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDQ2MC4yIDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNDYwLjIiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUi
IHk9IjExLjcwMDAwMSIgPlBhc3M8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iNDguNzUiIHk9IjExLjcwMDAwMSIgPjE6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjcyLjE1IiB5PSIxMS43MDAwMDEiID5FbnZpcm9ubWVudDwvdGV4dD4KICA8cG9s
eWdvbiBwb2ludHM9IjE3MS42LDMuOSAxNjMuOCw3LjggMTcxLjYsMTEuNzAwMDAxIiBjbGFzcz0i
ZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjE3MS42IiB5MT0iNy44IiB4Mj0iMTg3LjIw
MDAxIiB5Mj0iNy44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIxOTYuOTUwMDEiIHk9IjExLjcwMDAwMSIgPlJlZ2lzdGVyPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcyLjE1IiB5PSIyNy4zMDAwMDEiID5FbnZpcm9u
bWVudDwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9IjE3MS42LDE5LjUgMTYzLjgsMjMuNDAwMDAy
IDE3MS42LDI3LjMwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIx
NzEuNiIgeTE9IjIzLjQwMDAwMiIgeDI9IjE4Ny4yMDAwMSIgeTI9IjIzLjQwMDAwMiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTk2Ljk1MDAx
IiB5PSIyNy4zMDAwMDEiID5SZWdpc3RlcjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI3Mi4xNSIgeT0iNDIuOSIgPlR5cGVjaGVjazwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIxNTAuMTUwMDEiIHk9IjQyLjkiID5ib2RpZXM8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjA0Ljc1IiB5PSI0Mi45IiA+d2l0aDwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNDMuNzUiIHk9IjQyLjkiID5jb21wbGV0
ZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5PSI0Mi45IiA+
UGFzczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0OC43NSIgeT0iNDIu
OSIgPjI6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMxMy45NSIgeT0i
NDIuOSIgPm11dHVhbDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNjgu
NTUwMDIiIHk9IjQyLjkiID52aXNpYmlsaXR5ITwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIyNjcuMTUiIHk9IjExLjcwMDAwMSIgPmlzX2V2ZW4gOiAoSW50IC0mZ3Q7IEJv
b2wpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2Ny4xNSIgeT0iMjcu
MzAwMDAxIiA+aXNfb2RkICA6IChJbnQgLSZndDsgQm9vbCk8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 20–8: Two-Pass Environment Setup</figcaption>
</figure>

**Specification:**

Refactor `TypeChecker::visit(const ProgramAST&)` to perform two-pass
environment population. Pre-register all declared function signatures in
Pass 1, and verify function bodies in Pass 2.

``` cpp
void TypeChecker::visit(const ProgramAST& node) {
    // Pass 1: Forward declaration hoisting
    for (const auto& stmt : node.statements()) {
        if (const auto* let_stmt = dynamic_cast<const LetStmtAST*>(stmt.get())) {
            if (let_stmt->type_annotation().has_value()) {
                Type declared_t = convert_ast_type(*let_stmt->type_annotation());
                m_current_env->define(let_stmt->name(), declared_t, let_stmt->location());
            }
        }
    }
    // Pass 2: Full semantic verification
    for (const auto& stmt : node.statements()) {
        stmt->accept(*this);
    }
}
```

**Verification Tasks:**

1.  **Direct Recursion Verification:** Check the recursive `factorial`
    implementation. Verify that the self-referential call
    `factorial(n - 1)` resolves cleanly without diagnostic errors.
2.  **Mutual Recursion Verification:** Check the mutually recursive
    `is_even` and `is_odd` functions. Verify that both forward and
    backward function calls resolve correctly.
3.  **Non-Function Recursion Guard:** Check `let x = x + 1;` where `x`
    has not been defined previously. Verify that the checker flags an
    undeclared identifier error rather than allowing a variable to bind
    to itself.

#### Exercise 20.5: Hindley-Milner Type Inference and Unification (Algorithm W)

**Context and Theory:**

Writing explicit type annotations on every variable and lambda parameter
can be tedious. The **Hindley-Milner (HM)** type system deduces the most
general (principal) type for every expression without requiring
developer annotations:

$$\text{Identity Function: } \text{let } id = \text{fn } (x) \Rightarrow x \implies id : \forall \alpha. (\alpha \to \alpha)$$

Type inference operates by:

1.  Assigning fresh **Type Variables** ($\alpha, \beta, \dots$) to
    unannotated parameters and expressions.
2.  Generating type equality equations from operations (e.g. $x + 1$
    implies $\alpha_x = \text{Int}$).
3.  Solving equations via **Robinson's First-Order Unification
    Algorithm**:

$$\text{unify}(\tau_1, \tau_2) \implies \text{Substitution } \theta$$

A critical safety invariant is the **Occurs Check** (see Figure 20–9): a
type variable $\alpha$ cannot unify with a compound type containing
$\alpha$ (e.g. $\alpha \sim (\alpha \to \beta)$). Failing this check
indicates an infinite recursive type and must be rejected.

<figure id="fig:occurs-check">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MTQuOCIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUxNC44IDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTE0LjgiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUi
IHk9IjQyLjkiID5SZXN1bHQ6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjcyLjE1IiB5PSI0Mi45IiA+Q3ljbGU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMTE4Ljk1MDAwNSIgeT0iNDIuOSIgPmRldGVjdGVkITwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIxOTYuOTUwMDEiIHk9IjQyLjkiID5BYm9ydDwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNDMuNzUiIHk9IjQyLjkiID53aXRoPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI4Mi43NSIgeT0iNDIuOSIgPkluZmlu
aXRlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM1Mi45NSIgeT0iNDIu
OSIgPlR5cGU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzkxLjk1IiB5
PSI0Mi45IiA+RXJyb3IhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijku
NzUiIHk9IjExLjcwMDAwMSIgPkVxdWF0aW9uOiBhbHBoYSA9IChhbHBoYSAtJmd0OyBiZXRhKTwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5PSIyNy4zMDAwMDEi
ID5PY2N1cnMgQ2hlY2s6IERvZXMgJiMzOTthbHBoYSYjMzk7IGFwcGVhciBpbnNpZGUgJiMzOTso
YWxwaGEgLSZndDsgYmV0YSkmIzM5Oz8gWUVTITwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 20–9: Implementing the Occurs Check for Type
Inference</figcaption>
</figure>

**Specification:**

Implement a `TypeVariable` representation and a substitution environment
`std::unordered_map<uint64_t, Type>`. Construct a `TypeUnification`
class providing `unify(Type a, Type b)` with an occurs-check validator.

``` cpp
class TypeUnification {
public:
    static bool unify(Type& a, Type& b, std::unordered_map<uint64_t, Type>& subst) {
        Type t1 = resolve(a, subst);
        Type t2 = resolve(b, subst);
        if (t1 == t2) return true;
        if (t1.is_type_var()) return bind_var(t1.var_id(), t2, subst);
        if (t2.is_type_var()) return bind_var(t2.var_id(), t1, subst);
        if (t1.kind() == Type::Kind::Function && t2.kind() == Type::Kind::Function) {
            if (t1.param_types().size() != t2.param_types().size()) return false;
            for (size_t i = 0; i < t1.param_types().size(); ++i) {
                if (!unify(t1.param_types()[i], t2.param_types()[i], subst)) return false;
            }
            return unify(t1.return_type(), t2.return_type(), subst);
        }
        return false;
    }
private:
    static bool occurs_in(uint64_t var_id, const Type& target);
    static bool bind_var(uint64_t id, const Type& target, std::unordered_map<uint64_t, Type>& s);
    static Type resolve(const Type& t, const std::unordered_map<uint64_t, Type>& s);
};
```

**Verification Tasks:**

1.  **Simple Variable Deduction:** Unify a type variable $\alpha$ with
    `Int`. Verify that `resolve(alpha)` returns `Type::make_int()`.
2.  **Function Signature Synthesis:** Infer the type of
    `let double = fn (x) => x * 2;`. Verify that the parameter variable
    $\alpha$ unifies with `Int` and that the overall signature infers as
    `(Int -> Int)`.
3.  **Occurs Check Cycle Rejection:** Attempt to unify $\alpha$ with
    $(\alpha \to \text{Int})$. Verify that the occurs check triggers and
    rejects the self-referential cycle.

#### Exercise 20.6: Exhaustive Pattern Matching and Dead Arm Analysis

**Context and Theory:**

In a robust functional language, pattern matching over sum types,
booleans, and tuples must satisfy two semantic properties:

1.  **Exhaustiveness:** The match arms must cover every possible value
    in the domain. An incomplete match (e.g. matching `true` but
    omitting `false`) can cause a runtime crash.
2.  **Non-Redundancy (Dead Arm Detection):** No match arm should be
    unreachable. A match arm is dead if all values matching its pattern
    are already covered by preceding arms.

A wildcard `_` or bare identifier pattern matches any value
unconditionally. Therefore:

- If a match statement contains a wildcard arm, it is guaranteed to be
  **exhaustive**.
- Any arm positioned *after* an unconditional wildcard is completely
  **unreachable**.

```
 match flag with {
     true => 1,
     _    => 0,
     false => 2 // <=== DEAD ARM! Already shadowed by wildcard '_'!
 }
```

**Specification:**

Implement an `ExhaustivenessChecker` class that analyses `MatchExprAST`
nodes. Verify that boolean domains match both `true` and `false` (or
include a wildcard), and emit warning diagnostics for dead arms
positioned after wildcards.

``` cpp
class ExhaustivenessChecker {
public:
    static void verify_match(const MatchExprAST& match_node, Type target_type, 
                             DiagnosticReporter& reporter) {
        bool saw_wildcard = false;
        bool saw_true = false, saw_false = false;

        for (const auto& arm : match_node.arms()) {
            if (saw_wildcard) {
                reporter.report_warning(arm.pattern->location(), 
                                        "Unreachable match arm: preceding wildcard pattern already matches all values.");
            }
            if (arm.pattern->is_wildcard() || arm.pattern->is_variable()) {
                saw_wildcard = true;
            } else if (target_type == Type::make_bool()) {
                if (arm.pattern->matches_bool_literal(true)) saw_true = true;
                if (arm.pattern->matches_bool_literal(false)) saw_false = true;
            }
        }
        if (!saw_wildcard && target_type == Type::make_bool() && (!saw_true || !saw_false)) {
            reporter.report_error(match_node.location(), 
                                  "Non-exhaustive pattern match: missing case for '" + 
                                  std::string(!saw_true ? "true" : "false") + "'.");
        }
    }
};
```

**Verification Tasks:**

1.  **Non-Exhaustive Boolean Rejection:** Check
    `match is_ready with { true => 1 };`. Verify that the analyser
    rejects the expression, reporting that the `false` case is missing.
2.  **Wildcard Exhaustiveness:** Check
    `match is_ready with { true => 1, _ => 0 };`. Verify that the match
    passes with zero errors.
3.  **Dead Arm Warning:** Check
    `match x with { _ => "all", 42 => "forty-two" };`. Verify that the
    compiler reports a dead-arm warning pointing directly to `42`.

#### Exercise 20.7: Definite Assignment and Control-Flow Reachability

**Context and Theory:**

Reading uninitialised variables produces undefined behaviour in compiled
native code. Faberge enforces **Definite Assignment**: every variable
must be guaranteed to hold a valid value before any read access.

To enforce this, the semantic analyser tracks variable state across
branches:

- When a conditional `if cond then e1 else e2` executes, a variable $x$
  is definitely assigned after the conditional if and only if it is
  assigned in **both** the `then` branch and the `else` branch:

$$\text{Assigned}_{\text{after}}(x) = \text{Assigned}_{\text{then}}(x) \land \text{Assigned}_{\text{else}}(x)$$

If a variable is initialised in only one branch, it remains in an
uninitialised state following the conditional block (as shown in
Figure 20–10).

<figure id="fig:definite-assignment">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzMDZwdCIgaGVpZ2h0PSIxNjBwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzMDYuMTAgMTYwLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE1
NikiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gbGV0IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPmxldDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGN4PSIxMzQuOTIiIGN5PSItMTM0IiByeD0iNjIuOTciIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0
LWFuY2hvcj0ibWlkZGxlIiB4PSIxMzQuOTIiIHk9Ii0xMzEuNSIgZm9udC1mYW1pbHk9IkNvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPmxl
dCBtdXQgeDogSW50OzwvdGV4dD4KPC9nPgo8IS0tIHRoZW4gLS0+CjxnIGlkPSJub2RlMiIgY2xh
c3M9Im5vZGUiPgo8dGl0bGU+dGhlbjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSI1NS45MiIgY3k9Ii03NiIgcng9IjU1Ljg1IiByeT0iMTgiLz4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNTUuOTIiIHk9Ii03My41IiBmb250LWZhbWlseT0iQ291
cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+
dGhlbjogeCA9IDEwOzwvdGV4dD4KPC9nPgo8IS0tIGxldCYjNDU7Jmd0O3RoZW4gLS0+CjxnIGlk
PSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+bGV0JiM0NTsmZ3Q7dGhlbjwvdGl0bGU+Cjxw
YXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xMTIuNSwtMTE3LjFDMTA0LjQyLC0x
MTEuMzggOTUuMTcsLTEwNC44MiA4Ni41MiwtOTguNjkiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2si
IHN0cm9rZT0iYmxhY2siIHBvaW50cz0iODguMzEsLTk1LjY3IDc4LjEzLC05Mi43NCA4NC4yNywt
MTAxLjM4IDg4LjMxLC05NS42NyIvPgo8L2c+CjwhLS0gZWxzZSAtLT4KPGcgaWQ9Im5vZGUzIiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT5lbHNlPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgY3g9IjIxMy45MiIgY3k9Ii03NiIgcng9Ijg0LjM2IiByeT0iMTgiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjEzLjkyIiB5PSItNzMuNSIgZm9udC1mYW1pbHk9
IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAu
MDAiPmVsc2U6IC8vIHggdW5hc3NpZ25lZDwvdGV4dD4KPC9nPgo8IS0tIGxldCYjNDU7Jmd0O2Vs
c2UgLS0+CjxnIGlkPSJlZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+bGV0JiM0NTsmZ3Q7ZWxz
ZTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xNTcuMzUsLTEx
Ny4xQzE2NS4yMiwtMTExLjUyIDE3NC4yMSwtMTA1LjE1IDE4Mi42NywtOTkuMTYiLz4KPHBvbHln
b24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTg0Ljc2LC0xMDEuOTcgMTkw
Ljg5LC05My4zMyAxODAuNzEsLTk2LjI1IDE4NC43NiwtMTAxLjk3Ii8+CjwvZz4KPCEtLSByZWFk
IC0tPgo8ZyBpZD0ibm9kZTQiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnJlYWQ8L3RpdGxlPgo8ZWxs
aXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMTM0LjkyIiBjeT0iLTE4IiByeD0i
MzEuNCIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjEzNC45MiIgeT0i
LTE1LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3Nw
YWNlIiBmb250LXNpemU9IjEwLjAwIj5yZWFkIHg8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJz
dGFydCIgeD0iMTIuNzIiIHk9Ii0zNiIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUg
U2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlNlbWFudGljIEVycm9yITwv
dGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxMi43MiIgeT0iLTI1IiBmb250LWZh
bWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXpl
PSIxMC4wMCI+JiMzOTt4JiMzOTsgaXMgbm90PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3Rh
cnQiIHg9IjEyLjcyIiB5PSItMTQiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNh
bnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5hc3NpZ25lZCBvbjwvdGV4dD4K
PHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxMi43MiIgeT0iLTMiIGZvbnQtZmFtaWx5PSJD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAw
Ij5hbGwgcGF0aHMhPC90ZXh0Pgo8L2c+CjwhLS0gdGhlbiYjNDU7Jmd0O3JlYWQgLS0+CjxnIGlk
PSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+dGhlbiYjNDU7Jmd0O3JlYWQ8L3RpdGxlPgo8
cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNNzcuOTMsLTU5LjRDODcuMTgsLTUy
Ljg0IDk4LjA0LC00NS4xNSAxMDcuNzgsLTM4LjI0Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjEwOS44NSwtNDEuMDYgMTE1Ljk4LC0zMi40MyAxMDUuOCwt
MzUuMzUgMTA5Ljg1LC00MS4wNiIvPgo8L2c+CjwhLS0gZWxzZSYjNDU7Jmd0O3JlYWQgLS0+Cjxn
IGlkPSJlZGdlNCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+ZWxzZSYjNDU7Jmd0O3JlYWQ8L3RpdGxl
Pgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTkwLjY1LC01OC41QzE4MS42
NiwtNTIuMTMgMTcxLjMxLC00NC43OSAxNjEuOTksLTM4LjE5Ii8+Cjxwb2x5Z29uIGZpbGw9ImJs
YWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE2My45NywtMzUuMyAxNTMuNzksLTMyLjM3IDE1
OS45MiwtNDEuMDEgMTYzLjk3LC0zNS4zIi8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 20–10: Ensuring Definite Assignment (All
Paths)</figcaption>
</figure>

**Specification:**

Construct a `DefiniteAssignmentValidator` pass. Track variable
initialisation flags across control-flow splits (`if-then-else` and
block expressions). Raise semantic errors whenever an identifier is read
while in an unassigned state.

``` cpp
class DefiniteAssignmentValidator : public ASTVisitor {
public:
    void check_function_body(const ExprAST& body);
private:
    struct VarState { bool is_assigned{false}; };
    std::unordered_map<std::string, VarState> m_current_state;

    void visit(const IfExprAST& node) override {
        node.condition().accept(*this);
        auto state_before = m_current_state;
        node.then_branch().accept(*this);
        auto state_after_then = m_current_state;
        m_current_state = state_before;
        node.else_branch().accept(*this);
        auto state_after_else = m_current_state;
        // Intersection: must be assigned in BOTH branches
        for (auto& [var_name, var_state] : m_current_state) {
            var_state.is_assigned = state_after_then[var_name].is_assigned && 
                                    state_after_else[var_name].is_assigned;
        }
    }
};
```

**Verification Tasks:**

1.  **Unassigned Read Rejection:** Check
    `let mut a: Int; let b = a + 1;`. Verify that the checker flags `a`
    as an uninitialised read.
2.  **Branch Intersection Invariance:** Declare uninitialised
    `let mut x: Int;`. Assign `x = 10;` in the `then` branch, but leave
    `else` empty. Attempt to read `x` after the `if`. Verify that the
    read is rejected.
3.  **Symmetric Assignment Conformance:** Assign `x = 10;` in `then`,
    and `x = 20;` in `else`. Verify that reading `x` after the
    conditional succeeds with zero errors.

#### Exercise 20.8: Dead Code and Unreachable Expression Pruning

**Context and Theory:**

Dead code represents statements or expressions that can never be
executed under any control-flow path. Detecting dead code prevents
developer oversights and enables the compiler to prune dead branches
before target C++ emission.

Dead code arises primarily through:

1.  **Post-Return Statements:** Statements appearing in a block
    immediately after an unconditional `return` expression.
2.  **Statically Known Predicates:** Conditional expressions whose
    condition is a compile-time constant literal:
    - `if true then e1 else e2` $\implies e_2$ is dead code.
    - `if false then e1 else e2` $\implies e_1$ is dead code.

```
 {
     return 42;
     let dead_x = 100; // <=== UNREACHABLE CODE!
 }
```

**Specification:**

Create an `ASTDeadCodePruner` pass inheriting from `ASTVisitor`.
Identify statements appearing after returns and branches governed by
boolean literals. Emit diagnostic warnings and prune dead subtrees from
the AST.

``` cpp
class ASTDeadCodePruner : public ASTVisitor {
public:
    bool prune(std::unique_ptr<ExprAST>& expr_node) {
        if (auto* if_node = dynamic_cast<IfExprAST*>(expr_node.get())) {
            if (auto* lit = dynamic_cast<const LiteralExprAST*>(&if_node->condition())) {
                if (lit->token_type() == TokenType::Literal_Boolean) {
                    bool cond = (lit->value() == "true");
                    // Prune and promote active branch directly!
                    expr_node = cond ? std::move(if_node->extract_then()) 
                                     : std::move(if_node->extract_else());
                    return true;
                }
            }
        }
        return false;
    }
};
```

**Verification Tasks:**

1.  **Post-Return Warning:** Check `{ return 1; let x = 2; x }`. Verify
    that the analyser flags `let x = 2;` as unreachable dead code.
2.  **Constant Branch Pruning:** Run the pruner on
    `if true then 10 else 20`. Verify that the `IfExprAST` node is
    completely replaced by `LiteralExprAST(10)`.
3.  **Nested Dead Block Identification:** Test an outer `if false`
    guarding a complex block containing loops and declarations. Verify
    that all nested statements are identified as dead.

#### Exercise 20.9: Implicit Numeric Widening and Type Coercion

**Context and Theory:**

Strict static typing rejects mixed-mode arithmetic: adding an `Int` to a
`Float` fails because their bit-level representations differ (two's
complement integer vs IEEE-754 floating-point exponent/significand).

However, many languages permit **Safe Non-Lossy Widening** (see
Figure 20–11):

$$\text{Int} \subset \text{Float}$$

When evaluating binary arithmetic between an `Int` and a `Float`, the
semantic analyser:

1.  Recognises the mixed operands as compatible under widening coercion.
2.  Injects a synthetic `CastExprAST(operand, Float)` node into the AST
    around the integer operand.
3.  Synthesises the resulting expression type as `Float`.

Narrowing coercions (`Float` to `Int`) lose precision and are strictly
rejected.

<figure id="fig:safe-widening">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NjMiIGhlaWdo
dD0iMTI4LjgiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNjYzIDEyOC44Ij4KICA8c3R5
bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJl
Y3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEu
NTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVj
YXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAg
d2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5z
dmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50
Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2Ig
LmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZp
bGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXIt
ZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJr
ZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewog
IG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNs
ZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5f
Y2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3
Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9p
bnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgog
ICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNs
YXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVu
X2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAg
ICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+
CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjY2MyIgaGVpZ2h0
PSIxMjguOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9
IjExLjcwMDAwMSIgPk9yaWdpbmFsPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9Ijc5Ljk1MDAwNSIgeT0iMTEuNzAwMDAxIiA+QVNUOjwvdGV4dD4KICA8bGluZSB4MT0iMzQ3
LjEiIHkxPSIyNy4zMDAwMDEiIHgyPSIzNDcuMSIgeTI9IjUwLjciIGNsYXNzPSJzb2xpZCI+PC9s
aW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9Ijc0LjEiID5Db2Vy
Y2lvbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3OS45NTAwMDUiIHk9
Ijc0LjEiID5QYXNzOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1
IiB5PSIxMDUuMyIgPlJlc3VsdGluZzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI4Ny43NSIgeT0iMTA1LjMiID5UeXBlOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIxNTcuOTUiIHk9IjEwNS4zIiA+RmxvYXQ8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTU3Ljk1IiB5PSIxMS43MDAwMDEiID5CaW5hcnlFeHByKCspIFsg
TGVmdDogSW50KDEwKSwgUmlnaHQ6IEZsb2F0KDMuMTQpIF08L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTU3Ljk1IiB5PSI3NC4xIiA+QmluYXJ5RXhwcigrKSBbIExlZnQ6
IENhc3QmbHQ7RmxvYXQmZ3Q7KEludCgxMCkpLCBSaWdodDogRmxvYXQoMy4xNCkgXTwvdGV4dD4K
PC9zdmc+Cg==
" alt="" />
<figcaption>Figure 20–11: Safe Non-Lossy Widening</figcaption>
</figure>

**Specification:**

Implement a `can_coerce(Type from, Type to)` predicate. Extend
`TypeChecker::visit(const BinaryExprAST&)`: if an arithmetic operand
pair is `(Int, Float)` or `(Float, Int)`, rewrite the AST node to insert
a `CastExprAST` node wrapping the integer operand and assign the result
type `Float`.

``` cpp
class NumericCoercionEngine {
public:
    static bool try_coerce_binary(std::unique_ptr<ExprAST>& left, Type& left_t,
                                  std::unique_ptr<ExprAST>& right, Type& right_t) {
        if (left_t == Type::make_int() && right_t == Type::make_float()) {
            left = std::make_unique<CastExprAST>(left->location(), std::move(left), Type::make_float());
            left_t = Type::make_float();
            return true;
        }
        if (left_t == Type::make_float() && right_t == Type::make_int()) {
            right = std::make_unique<CastExprAST>(right->location(), std::move(right), Type::make_float());
            right_t = Type::make_float();
            return true;
        }
        return false;
    }
};
```

**Verification Tasks:**

1.  **Left-Operand Widening:** Check `let res = 10 + 3.14;`. Verify that
    the left operand is wrapped with a cast to `Float` and that `res`
    infers type `Float`.
2.  **Right-Operand Widening:** Check `let res = 2.5 * 4;`. Verify that
    the right operand is coerced and that the expression evaluates to
    `Float`.
3.  **Narrowing Rejection:** Attempt to assign a float to an explicitly
    annotated integer: `let x: Int = 3.14;`. Verify that the coercion
    engine refuses to implicitly narrow `Float` to `Int`.

#### Exercise 20.10: Unused Symbol and Variable Shadowing Warnings

**Context and Theory:**

High-quality compilers act as static analysis linters, warning
developers about suspicious code patterns that are syntactically legal
but indicative of logic bugs:

1.  **Unused Identifiers:** A variable is declared and initialised, but
    its identifier is never read within its scope.
2.  **Variable Shadowing:** An identifier in an inner scope shares the
    exact name of an active identifier in an outer enclosing scope. This
    can lead to subtle bugs where an author mistakenly assumes they are
    modifying the outer variable.

To support this without false positives, modern compilers allow a
leading underscore prefix (e.g. `_temp` or `_unused`) to signal
intentional non-use.

```
 let x = 10;
 {
     let x = 20; // WARNING: Declaration of 'x' shadows outer binding at line 1!
 }
 // WARNING: Variable 'x' declared at line 1 is never read!
```

**Specification:**

Add a `size_t read_count` to the `Symbol` structure in `SymbolTable`.
Increment the counter whenever `lookup()` is called during expression
checking. When exiting a scope (`pop_scope()`), inspect all bindings and
report unused variables (skipping identifiers prefixed with `_`).

``` cpp
class LinterEnvironment : public Environment {
public:
    void check_unused_symbols(DiagnosticReporter& reporter) {
        for (const auto& [name, sym] : m_bindings) {
            if (sym.read_count == 0 && !name.starts_with('_')) {
                reporter.report_warning(sym.location, 
                                        "Unused variable '" + name + "'. Prefix with '_' if intentional.");
            }
        }
    }
};
```

**Verification Tasks:**

1.  **Unused Variable Detection:** Check
    `let total = 100; let rate = 0.05; return rate;`. Verify that the
    compiler reports a warning for `total` and zero warnings for `rate`.
2.  **Underscore Suppression:** Check `let _placeholder = 42;`. Verify
    that the leading underscore suppresses the unused variable warning.
3.  **Shadowing Detection:** Check
    `let x = 10; { let x = 20; return x; }`. Verify that the inner
    declaration triggers a variable-shadowing warning pointing back to
    the line coordinate of the outer `x`.

#### Exercise 20.11: Function Overload Resolution and Candidate Ranking

**Context and Theory:**

Many programming languages support **Function Overloading**: multiple
functions sharing the same name but possessing different parameter
signatures:

```
let print = fn (x: Int): Void => ... ;
let print = fn (s: String): Void => ... ;
```

When a call expression `print(arg)` is analysed, the type checker must
resolve the call (see Figure 20–12):

1.  **Candidate Gathering:** Retrieve all functions matching the callee
    identifier in scope $\Gamma$.
2.  **Arity Filtering:** Discard candidates whose parameter count does
    not match argument count.
3.  **Type Compatibility Matching:** Discard candidates whose parameter
    types cannot accept the argument types (accounting for exact matches
    vs widened coercions).
4.  **Best-Match Ranking:** Rank surviving candidates. An exact type
    match scores higher than a coerced match. If two candidates tie with
    identical scores, raise an **Ambiguous Call Error**.

<figure id="fig:function-overloading">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MDciIGhlaWdo
dD0iMTQ0LjQwMDAxIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUwNyAxNDQuNDAwMDEi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNw
eDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJh
bnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoK
LnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmls
bCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlh
bW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFy
a2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0K
Ci5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Bl
bl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtl
ci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29w
ZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNi
aWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJy
b3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9Ijci
IG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9s
eWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAg
ICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9
IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1y
ZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3Bv
bHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2Np
cmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3
Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQi
IGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgog
IDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTA3
IiBoZWlnaHQ9IjE0NC40MDAwMSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjkuNzUiIHk9IjExLjcwMDAwMSIgPkNhbmRpZGF0ZXM6IFsxXSBwcmludChJbnQpLCBbMl0g
cHJpbnQoRmxvYXQpLCBbM10gcHJpbnQoU3RyaW5nKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI5Ljc1IiB5PSI0Mi45IiA+Q2FsbDogICAgICAgcHJpbnQoNDIpPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9Ijc0LjEiID5SYW5raW5n
OiAgICBDYW5kaWRhdGUgWzFdIChFeGFjdCBNYXRjaDogU2NvcmUgMCk8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTAzLjM1MDAwNiIgeT0iODkuNzAwMDA1IiA+Q2FuZGlk
YXRlIFsyXSAoQ29lcmNpb24gTWF0Y2g6IFNjb3JlIDEpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjEyMC45IiA+V2lubmVyOiAgICAgQ2FuZGlkYXRlIFsx
XSBzZWxlY3RlZCE8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 20–12: Candidate Selection in Function
Overloading</figcaption>
</figure>

**Specification:**

Construct an `OverloadResolver` class. Store overloaded symbols in the
environment as `std::vector<Symbol>`. Implement candidate scoring and
best-match selection during `CallExprAST` verification.

``` cpp
class OverloadResolver {
public:
    static Symbol resolve_call(const std::string& name, const std::vector<Type>& arg_types,
                               const std::vector<Symbol>& candidates, SourceLocation loc) {
        std::vector<std::pair<int, Symbol>> viable;
        for (const auto& cand : candidates) {
            int score = score_candidate(cand.type.param_types(), arg_types);
            if (score >= 0) viable.emplace_back(score, cand);
        }
        if (viable.empty()) throw std::runtime_error("No matching overload for '" + name + "'.");
        std::sort(viable.begin(), viable.end(), [](auto& a, auto& b) { return a.first < b.first; });
        if (viable.size() > 1 && viable[0].first == viable[1].first) {
            throw std::runtime_error("Ambiguous call to overloaded function '" + name + "'.");
        }
        return viable[0].second;
    }
private:
    static int score_candidate(const std::vector<Type>& params, const std::vector<Type>& args);
};
```

**Verification Tasks:**

1.  **Exact Overload Selection:** Register `f(Int)` and `f(String)`.
    Check `f(10)`. Verify that the resolver binds the call directly to
    the `f(Int)` overload.
2.  **Widened Overload Selection:** Register `g(Float)` and `g(String)`.
    Check `g(10)`. Verify that `10` is coerced to `Float` and binds to
    `g(Float)`.
3.  **Ambiguity Rejection:** Register `h(Int, Float)` and
    `h(Float, Int)`. Check `h(10, 20)`. Verify that the resolver detects
    an ambiguous tie and raises a compile-time diagnostic.

#### Exercise 20.12: Static Escape and Closure Capture Lifetime Analysis

**Context and Theory:**

In a functional language, a lambda can reference variables declared in
its enclosing scope (**Free Variables**):

```
let make_multiplier = fn (factor: Int): (Int -> Int) => {
    fn (x: Int): Int => x * factor // 'factor' is captured from outer scope!
};
```

Because our compiler targets **C++**, the lifetime of captured variables
is critical:

- If a lambda is evaluated only locally, it can capture variables by
  reference (`[&]`).
- If a lambda **escapes** its defining function (e.g. by being returned
  or assigned to a global), capturing local stack variables by reference
  creates a dangling pointer when the outer frame is destroyed.

The semantic analyser must compute the **Capture Set** for every lambda:

$$\text{Captured}(\lambda) = \text{FreeVars}(\lambda) \cap \text{EnclosingLocals}$$

Lambdas that escape their enclosing scope (as shown in Figure 20–13)
must be flagged so the downstream C++ emitter captures variables by
value (`[=]`) or inside heap-allocated `std::shared_ptr` wrapper.

<figure id="fig:return-inner">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NTUuMiIgaGVp
Z2h0PSI4MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2NTUuMiA4MiI+CiAgPHN0eWxl
Pi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2JvYiByZWN0
LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRoOiAxLjU7
CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1saW5lY2Fw
OiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQgewogIHdo
aXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9Cgouc3Zn
Ym9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3BhcmVudDsK
fQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3ZnYm9iIC5m
aWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZpbGw6IHRy
YW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFya2VyLWVu
ZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25kIHsKICBt
YXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2NpcmNsZSk7
Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9jaXJjbGUg
ewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19vcGVuX2Np
cmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIgdmlld0Jv
eD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50
cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2Vy
V2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgog
ICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdvbj4KICAg
IDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZY
PSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1
dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFz
cz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0ib3Blbl9j
aXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3
IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNp
cmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4
IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9y
aWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9
IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgog
IDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2NTUuMiIgaGVpZ2h0
PSI4MiI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iNTQuNjAwMDAyLDM1LjEwMDAwMiA2Mi40
LDM5IDU0LjYwMDAwMiw0Mi45IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzIuMTUiIHk9IjQyLjkiID5Jbm5lcjwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMTguOTUwMDA1IiB5PSI0Mi45IiA+TGFtYmRhOjwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxODEuMzUiIHk9IjQyLjkiID5S
ZWZlcmVuY2VzPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2Ny4xNSIg
eT0iNDIuOSIgPiYjMzk7ZmFjdG9yJiMzOTs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iNzIuMTUiIHk9IjU4LjUiID5JczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSI5NS41NSIgeT0iNTguNSIgPklubmVyPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjE0Mi4zNSIgeT0iNTguNSIgPkxhbWJkYTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxOTYuOTUwMDEiIHk9IjU4LjUiID5yZXR1cm5lZD88L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjc0Ljk1IiB5PSI1OC41IiA+WUVT
PC90ZXh0PgogIDxsaW5lIHgxPSIzMDQuMiIgeTE9IjU0LjYwMDAwMiIgeDI9IjMxOS44MDAwMiIg
eTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIz
MTkuODAwMDIsNTAuNyAzMjcuNiw1NC42MDAwMDIgMzE5LjgwMDAyLDU4LjUiIGNsYXNzPSJmaWxs
ZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMzcuMzUiIHk9
IjU4LjUiID4mIzM5O2ZhY3RvciYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjQwNy41NTAwMiIgeT0iNTguNSIgPkVTQ0FQRVMhPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjQ3Ny43NSIgeT0iNTguNSIgPk11c3Q8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTE2Ljc1IiB5PSI1OC41IiA+Y2FwdHVyZTwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1NzkuMTUiIHk9IjU4LjUiID5ieTwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2MDIuNTUiIHk9IjU4LjUiID52YWx1
ZSE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0iMTEuNzAw
MDAxIiA+U2NvcGU6IG1ha2VfbXVsdGlwbGllciAoTG9jYWw6IGZhY3Rvcik8L3RleHQ+CiAgPGc+
CiAgICA8bGluZSB4MT0iMjcuMzAwMDAxIiB5MT0iMTUuNiIgeDI9IjI3LjMwMDAwMSIgeTI9IjYy
LjQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjI3LjMwMDAwMSIgeTE9IjM5
IiB4Mj0iNTQuNjAwMDAyIiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KPC9z
dmc+Cg==
" alt="" />
<figcaption>Figure 20–13: Inner Lambda Escape</figcaption>
</figure>

**Specification:**

Create an `EscapeAnalysisVisitor` class inheriting from `ASTVisitor`.
For every `LambdaExprAST`, compute its free variables, identify whether
the lambda escapes its declaring stack frame, and record its capture
list for the C++ code generator.

``` cpp
class EscapeAnalysisVisitor : public ASTVisitor {
public:
    struct LambdaMetadata {
        bool escapes_local_frame{false};
        std::unordered_set<std::string> captured_variables;
    };
    void analyse(const ProgramAST& root);
    const LambdaMetadata& get_metadata(const LambdaExprAST* lambda) const;
private:
    std::vector<std::unordered_set<std::string>> m_scope_stack;
    std::unordered_map<const LambdaExprAST*, LambdaMetadata> m_lambda_table;
    void visit(const VariableExprAST& node) override;
    void visit(const LambdaExprAST& node) override;
};
```

**Verification Tasks:**

1.  **Local Free Variable Capture:** Check a lambda accessing an outer
    variable `x`. Verify that `x` is correctly added to the lambda's
    capture set.
2.  **Escaping Lambda Identification:** Check a function returning an
    inner closure referencing a parameter. Verify that the inner lambda
    is marked with `escapes_local_frame = true`.
3.  **Pure Function Invariance:** Check a lambda referencing only its
    own parameters and global constants. Verify that its capture set is
    empty (`size() == 0`).